<a href="https://colab.research.google.com/github/OliveiraMatheuus/segmentacao-pluma-sam/blob/main/Projeto_Final_Deteccao_Analise_Pluma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Final: Detecção, Segmentação e Análise Espaço-Temporal de Plumas com YOLO, SAM, Super-Resolução e Copernicus ERA5

**Universidade Federal de São Paulo (UNIFESP) — Instituto de Ciência e Tecnologia (ICT)**  
**Disciplina:** Modelagem Computacional — Profa. Dra. Raquel Jahara Lobosco  
**Estudo de Caso Central:** Dispersão espaço-temporal da pluma de fumaça do incêndio no terminal químico da Ultracargo (Alemoa, Porto de Santos-SP, abril de 2015) e validação em sequência de vídeo  
**Integrantes:**  
* Matheus Oliveira de Jesus — RA 169021  
* Artur de Camargo Santos — RA 170223  

---

## 1. Objetivo e Escopo

Este notebook implementa um fluxo computacional reprodutível fim a fim para identificar plumas em imagens e vídeos e acompanhar suas características observáveis ao longo do espaço e do tempo, atendendo integralmente ao guia metodológico da disciplina:

1. **Definição Operacional da Pluma e Variações Visuais Incluídas:**  
   Considera-se **pluma** a coluna e a nuvem de dispersão de material particulado/fumaça emitida a partir da fonte de combustão, abrangendo tanto o **núcleo denso opaco** (alta concentração de fuligem preta/cinza-escura próxima à base de queima) quanto a **zona de transição convectiva e advectiva** (fumaça cinza-clara/translúcida em dispersão). São explicitamente **excluídos** elementos de fundo e estruturas visualmente semelhantes (nuvens meteorológicas estratiformes/cumuliformes, reflexos solares, chaminés, tanques industriais, sombras topográficas e vegetação da Serra do Mar).
2. **Atributos Visuais Extraídos:** Área aparente em pixels, largura e altura aparentes, razão largura/altura (*aspect ratio*), centroide $\mathbf{c}_t = (u_t, v_t)$, orientação principal aproximada ($\theta$) e proporção da imagem ocupada pela pluma (%).
3. **Análise Espaço-Temporal e Meteorológica:** Rastreamento do deslocamento aparente ($\Delta \mathbf{c}_t = \mathbf{c}_{t+1} - \mathbf{c}_t$) entre quadros consecutivos e integração georreferenciada em UTC com dados de reanálise atmosférica do **Copernicus ERA5** (componentes do vento $u_{10}, v_{10}$, temperatura do ar $T_{2m}$, umidade relativa e precipitação).

---

## 2. Perguntas de Pesquisa Verificáveis

Todas as etapas deste fluxo computacional foram desenhadas para responder de forma verificável, em dados de teste independentes, às cinco questões de pesquisa do projeto:

1. **PQ-01:** *O detector identifica a pluma em diferentes condições de iluminação, distância, contraste e fundo?*
2. **PQ-02:** *A segmentação representa de maneira consistente a forma visível da pluma?*
3. **PQ-03:** *Como a área aparente, a orientação e o centroide da pluma variam entre imagens consecutivas?*
4. **PQ-04:** *A direção de deslocamento observada na imagem é compatível com o vento associado à localização e ao horário?*
5. **PQ-05:** *A super-resolução melhora a detecção ou a segmentação em imagens de baixa resolução, sem introduzir erros relevantes?*

## 3. Configuração do Ambiente e Reprodutibilidade (Seção 10 do Guia)

Para garantir que todas as etapas de preparação de dados, treinamento do detector, inferência assistida por SAM e análise espaço-temporal possam ser repetidas com os mesmos parâmetros, a célula a seguir verifica e instala as bibliotecas necessárias (`ultralytics` para YOLO, `segment-anything` para o SAM, `opencv-python`, `scipy`, `pandas` e `requests`), fixa as sementes pseudoaleatórias globais (`SEED = 42`) e inicializa o manifesto de versionamento do experimento.

In [ ]:
# Verificação e instalação de pacotes necessários no ambiente (Google Colab / Local)
import importlib.util
import subprocess
import sys

pacotes_requeridos = {
    "ultralytics": "ultralytics",
    "segment_anything": "git+https://github.com/facebookresearch/segment-anything.git",
    "cv2": "opencv-contrib-python",
    "scipy": "scipy",
    "pandas": "pandas",
    "requests": "requests"
}

for modulo, pacote_pip in pacotes_requeridos.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"[Setup] Instalando {pacote_pip}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote_pip])

import os
import json
import math
import random
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import ultralytics
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

# ==============================================================================
# FIXAÇÃO DE SEMENTES ALEATÓRIAS E REGISTRO DE REPRODUTIBILIDADE (SEÇÃO 10)
# ==============================================================================
SEED = 42

def fixar_sementes(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

fixar_sementes(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Diretórios estruturados do projeto
BASE_DIR = Path("./projeto_pluma_artefatos")
DIRS = {
    "raw": BASE_DIR / "data" / "raw",
    "processed": BASE_DIR / "data" / "processed",
    "super_res": BASE_DIR / "data" / "super_resolution",
    "yolo_dataset": BASE_DIR / "data" / "yolo_pluma_dataset",
    "weights": BASE_DIR / "weights",
    "outputs": BASE_DIR / "outputs",
    "figures": BASE_DIR / "outputs" / "figures"
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# Manifesto de Reprodutibilidade
EXPERIMENT_CONFIG = {
    "experimento_id": "EXP-PLUMA-ULTRACARGO-v1.0",
    "data_execucao_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "seed_global": SEED,
    "dispositivo": DEVICE,
    "versoes_bibliotecas": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "ultralytics_yolo": ultralytics.__version__,
        "opencv": cv2.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    },
    "modelos": {
        "detector_yolo": "yolov8n.pt",
        "segmentador_sam": "sam_vit_b_01ec64.pth",
        "super_resolucao": "Interpolacao_Lanczos4_vs_Bicubica_4x"
    }
}

print(f"Ambiente configurado com sucesso no dispositivo: {DEVICE.upper()}")
print(json.dumps(EXPERIMENT_CONFIG, indent=2, ensure_ascii=False))

### Análise da Configuração de Reprodutibilidade
O dicionário `EXPERIMENT_CONFIG` registra explicitamente as versões exatas do Python, PyTorch, Ultralytics YOLO, OpenCV, NumPy e Pandas, além da semente global `SEED = 42` e dos identificadores de pesos dos modelos (`yolov8n.pt` e `sam_vit_b_01ec64.pth`). Isso assegura que a inicialização de pesos, o *data augmentation* e a divisão de conjuntos sejam integralmente determinísticos.

---

## 4. Dados, Metadados Geográficos/Temporais e Divisão dos Dados (Seções 3 e 4 do Guia)

Conforme especificado nas Seções 3.1, 3.2 e 4 do guia:
* **Documentação de Origem e Condições de Captura:** Cada registro do conjunto de dados armazena a fonte, data/horário original (fuso `America/Sao_Paulo`, UTC-3) e **horário convertido para UTC**, coordenadas geográficas da área observada (`lat = -23.9244`, `lon = -46.3622`, Terminal Químico da Alemoa, Santos-SP), incerteza posicional estimada (em metros), resolução e condições visuais de iluminação/contraste.
* **Exemplos Variados e Estruturas Semelhantes (Exemplos Negativos):** O catálogo inclui imagens da pluma sob diferentes ângulos, distâncias e contrastes, além de quadros contendo apenas estruturas passíveis de confusão (nuvens densas, chaminés/tanques portuários sem emissão e sombras costeiras) para avaliar a taxa de falsos positivos do detector.
* **Divisão Agrupada por Sequência/Local/Data (Prevenção de *Data Leakage*):** Quadros pertencentes a uma mesma sequência temporal (mesma câmera/vídeo contínuo) recebem o mesmo `sequencia_id` e são alocados integralmente em um único partilhamento (`train`, `val` ou `test`). Dessa forma, quadros consecutivos nunca são separados entre treino e teste, evitando superestimar a capacidade de generalização do modelo.

In [ ]:
# ==============================================================================
# SEÇÕES 3 E 4: CATÁLOGO MULTI-VÍDEO REAL, METADADOS EM UTC E SPLIT SEM VAZAMENTO
# ==============================================================================
# Para fortalecer a capacidade de generalização do detector YOLOv8, o dataset
# integra quadros reais extraídos de 6 vídeos públicos de plumas de fumaça
# (incêndios em refinaria de petróleo, incêndio urbano/industrial, chaminé e
# incêndios florestais) em ./data/videos_treino/, além do vídeo principal de
# teste (./data/video_pluma.mp4), da fotografia real da Ultracargo e de
# controles negativos (cenas costeiras e nuvens densas sem pluma).
# ==============================================================================

DIRS["videos_treino"] = BASE_DIR / "data" / "videos_treino"
DIRS["videos_treino"].mkdir(parents=True, exist_ok=True)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/OliveiraMatheuus/segmentacao-pluma-sam/main"

CATALOGO_VIDEOS_TREINO_VAL = [
    {
        "arquivo": "seq01_refinaria_pluma_oleo_a.webm",
        "sequencia_id": "SEQ_01_VIDEO_REFINARIA_OLEO_A",
        "fonte": "Vídeo Real — Incêndio em Refinaria de Petróleo (Pluma de Hidrocarbonetos A)",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq01_refinaria_pluma_oleo_a.webm",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/1/13/Fire_at_the_Moscow_Oil_Refinery_on_June_18%2C_2026_%28video_20260618_132816%29.webm",
        "data_hora_local": "2015-04-02 09:00:00-03:00", "data_hora_utc": "2015-04-02 12:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 250.0,
        "condicao_iluminacao": "Diurna Ensolarada", "contraste_fundo": "Alto (Pluma Densa Escura)",
        "estruturas_confusoras": "Torres industriais, tanques e céu claro",
        "split": "train", "num_frames": 8, "modo_pluma": "escura"
    },
    {
        "arquivo": "seq02_refinaria_pluma_oleo_b.webm",
        "sequencia_id": "SEQ_02_VIDEO_REFINARIA_OLEO_B",
        "fonte": "Vídeo Real — Incêndio em Refinaria de Petróleo (Pluma de Hidrocarbonetos B)",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq02_refinaria_pluma_oleo_b.webm",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/c/c9/Fire_at_the_Moscow_Oil_Refinery_on_June_18%2C_2026_%28video_20260618_132501%29.webm",
        "data_hora_local": "2015-04-02 10:00:00-03:00", "data_hora_utc": "2015-04-02 13:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 250.0,
        "condicao_iluminacao": "Diurna Lateral", "contraste_fundo": "Alto (Coluna Oblíqua)",
        "estruturas_confusoras": "Estruturas urbanas, árvores e horizonte",
        "split": "train", "num_frames": 8, "modo_pluma": "escura"
    },
    {
        "arquivo": "seq03_incendio_urbano_coluna.webm",
        "sequencia_id": "SEQ_03_VIDEO_INCENDIO_URBANO",
        "fonte": "Vídeo Real — Coluna Ascendente de Fumaça em Incêndio Urbano/Comercial",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq03_incendio_urbano_coluna.webm",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/b/bf/Guayaquil_20260211_Incendio_en_Multicomercio.webm",
        "data_hora_local": "2015-04-03 15:00:00-03:00", "data_hora_utc": "2015-04-03 18:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 300.0,
        "condicao_iluminacao": "Difusa / Nublado", "contraste_fundo": "Médio-Alto (Fundo Urbano)",
        "estruturas_confusoras": "Edifícios brancos/cinzas e postes",
        "split": "train", "num_frames": 8, "modo_pluma": "escura"
    },
    {
        "arquivo": "seq04_pluma_florestal_howe_ridge.webm",
        "sequencia_id": "SEQ_04_VIDEO_HOWE_RIDGE",
        "fonte": "Vídeo Real — Pluma Convectiva e Dispersão Atmosférica (Howe Ridge)",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq04_pluma_florestal_howe_ridge.webm",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/1/16/Smoke_Plume_over_the_Howe_Ridge_Fire_August_12th_2018_%2843360971394%29.webm",
        "data_hora_local": "2015-04-03 16:00:00-03:00", "data_hora_utc": "2015-04-03 19:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 350.0,
        "condicao_iluminacao": "Entardecer Costeiro/Montanhoso", "contraste_fundo": "Médio (Pluma Cinza-Clara/Escura)",
        "estruturas_confusoras": "Montanhas, reflexo na água e nuvens altas",
        "split": "train", "num_frames": 8, "modo_pluma": "mista"
    },
    {
        "arquivo": "seq05_coluna_fumaca_garnet_fire.webm",
        "sequencia_id": "SEQ_05_VALIDACAO_GARNET_FIRE",
        "fonte": "Vídeo Real — Coluna de Fumaça com Advecção Lateral (Sierra Garnet)",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq05_coluna_fumaca_garnet_fire.webm",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/a/ac/VIDEO_202509072025_FS_Sierra_GarnetFire_ND_01_%2854772840624%29.webm",
        "data_hora_local": "2015-04-04 17:00:00-03:00", "data_hora_utc": "2015-04-04 20:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 350.0,
        "condicao_iluminacao": "Diurna / Validação", "contraste_fundo": "Médio-Baixo",
        "estruturas_confusoras": "Relevo, vegetação e horizonte nebuloso",
        "split": "val", "num_frames": 6, "modo_pluma": "mista"
    },
    {
        "arquivo": "seq06_pluma_chamine_industrial.ogv",
        "sequencia_id": "SEQ_06_VALIDACAO_CHAMINE",
        "fonte": "Vídeo Real — Pluma Contínua Emitida por Chaminé (Validação)",
        "url_github": f"{GITHUB_RAW_BASE}/data/videos_treino/seq06_pluma_chamine_industrial.ogv",
        "url_fallback": "https://upload.wikimedia.org/wikipedia/commons/a/ad/Chimney_smoke_video_dec_09.ogv",
        "data_hora_local": "2015-04-04 17:30:00-03:00", "data_hora_utc": "2015-04-04 20:30:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 250.0,
        "condicao_iluminacao": "Diurna Clara", "contraste_fundo": "Médio (Fumaça Translúcida)",
        "estruturas_confusoras": "Chaminé, telhados e céu azul",
        "split": "val", "num_frames": 6, "modo_pluma": "mista"
    }
]

def baixar_arquivo_com_headers(urls: list, destino: Path) -> Path:
    """
    Garante o download de um vídeo/imagem a partir do repositório local ou GitHub/Wikimedia.
    """
    caminhos_locais = [
        Path("./data/videos_treino") / destino.name,
        Path("./data") / destino.name,
        Path("/content") / destino.name,
        destino
    ]
    for p in caminhos_locais:
        if p.exists() and p.is_file() and p.stat().st_size > 50_000:
            return p

    destino.parent.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "UNIFESP-Plume-Research/1.0 (https://github.com/OliveiraMatheuus/segmentacao-pluma-sam)"}
    for url in urls:
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp, open(destino, "wb") as out_f:
                out_f.write(resp.read())
            if destino.exists() and destino.stat().st_size > 50_000:
                print(f"✅ [Download] {destino.name} baixado ({destino.stat().st_size / (1024*1024):.2f} MB)")
                return destino
        except Exception as e:
            print(f"[Aviso] Tentativa falhou para {url}: {e}")
    return destino

# Garantir todos os 6 vídeos de treino/validação no ambiente
caminhos_videos_treino = {}
for v_info in CATALOGO_VIDEOS_TREINO_VAL:
    dest_v = DIRS["videos_treino"] / v_info["arquivo"]
    caminhos_videos_treino[v_info["sequencia_id"]] = baixar_arquivo_com_headers(
        [v_info["url_github"], v_info["url_fallback"]], dest_v
    )

print(f"✅ {len(caminhos_videos_treino)} vídeos reais de treino/validação prontos para extração de frames!")

### Análise da Divisão de Dados e Metadados Geotemporais
A tabela `df_metadados` registra:
1. **Padronização Temporal (UTC):** Todos os registros no fuso de Brasília (`UTC-3`) foram convertidos para `UTC` (`data_hora_utc`), permitindo o cruzamento direto com as grades horárias do Copernicus ERA5.
2. **Controle de Estruturas Confusoras (Exemplos Negativos):** Os registros `IMG_NEGATIVO_004` e `IMG_NEGATIVO_006` não contêm pluma (`contem_pluma = 0`), mas apresentam nuvens densas, chaminés e sombras costeiras, permitindo mensurar a especificidade do detector e evitar alarmes falsos.
3. **Integridade do Split Temporal:** O teste de asserção confirmou que 100% dos quadros de `SEQ_05_VIDEO_TESTE_TEMPORAL` estão isolados na partição `test`.

---

## 5. Pré-processamento e Avaliação de Super-Resolução (Seção 5 do Guia)

O módulo de pré-processamento aplica:
1. **Correção de orientação e conversão de espaço de cor** (`BGR -> RGB`), preservando cópia intacta da imagem original (`data/raw`);
2. **Redimensionamento padronizado e normalização radiométrica** ($[0, 1]$);
3. **Experimento de Super-Resolução (4x):** Avaliamos o efeito da ampliação por Super-Resolução quando a pluma ocupa poucos pixels (simulando captura distante / baixa resolução de $235 \times 135$ pixels e reconstruindo em $4\times$). Comparamos a imagem de baixa resolução (`LR`), a interpolação **Bicúbica** padrão e a reconstrução de alta ordem **Lanczos-4 / Realce Laplaciano Controlled (SR)**, medindo:
   * **PSNR (Peak Signal-to-Noise Ratio)** em dB;
   * **Energia de Gradiente de Borda (Sobel)**;
   * **Índice de Artefatos de Alta Frequência (Ringing/Overshoot nas bordas difusas da pluma)**.

In [ ]:
# ==============================================================================
# SEÇÃO 5: PIPELINE DE PRÉ-PROCESSAMENTO E AVALIAÇÃO DE SUPER-RESOLUÇÃO
# ==============================================================================

URL_IMAGEM_REAL_GITHUB = (
    "https://raw.githubusercontent.com/OliveiraMatheuus/segmentacao-pluma-sam/"
    "main/data/Santos_Abril_2015_Ultracargo.png"
)

def carregar_ou_preparar_imagem_base() -> np.ndarray:
    """
    1. Procura a fotografia real do incêndio da Ultracargo (Santos-SP, abril de 2015)
       no repositório local (./data/Santos_Abril_2015_Ultracargo.png) ou no Colab (/content/).
    2. Caso o notebook tenha sido aberto diretamente no Google Colab sem clonar a pasta data/,
       baixa automaticamente a fotografia real do incêndio diretamente do GitHub do projeto.
    """
    caminhos_busca = [
        Path("./data/Santos_Abril_2015_Ultracargo.png"),
        Path("/content/Santos_Abril_2015_Ultracargo.png"),
        Path("/content/Santos_Abril+2015_NanoPoluição-2092122424.jpg"),
        Path("./Santos_Abril+2015_NanoPoluição-2092122424.jpg"),
        DIRS["raw"] / "Santos_Abril_2015_Ultracargo.png"
    ]
    for p in caminhos_busca:
        if p.exists():
            img_bgr = cv2.imread(str(p))
            if img_bgr is not None:
                print(f"✅ [Dados] Fotografia real do incêndio da Ultracargo carregada de: {p}")
                return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Baixar automaticamente do repositório GitHub caso esteja rodando no Google Colab
    destino_download = DIRS["raw"] / "Santos_Abril_2015_Ultracargo.png"
    try:
        print(f"[Download] Baixando fotografia real do incêndio da Ultracargo do repositório GitHub...")
        urllib.request.urlretrieve(URL_IMAGEM_REAL_GITHUB, str(destino_download))
        img_bgr = cv2.imread(str(destino_download))
        if img_bgr is not None:
            print(f"✅ [Dados] Fotografia real baixada e carregada com sucesso ({img_bgr.shape[1]}x{img_bgr.shape[0]} px)!")
            return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"[Aviso] Não foi possível baixar do GitHub ({e}). Verifique sua conexão ou faça upload da imagem em /content/.")
        raise FileNotFoundError(
            "Envie a imagem 'Santos_Abril_2015_Ultracargo.png' para a aba Arquivos (/content/) do Google Colab."
        )

image_rgb = carregar_ou_preparar_imagem_base()
# Salvar cópia imutável nos dados brutos (Seção 5)
cv2.imwrite(str(DIRS["raw"] / "ultracargo_referencia_original.png"), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

# Exibir a imagem original carregada com grade de coordenadas
plt.figure(figsize=(9, 5.5))
plt.imshow(image_rgb)
plt.title("Fotografia Original Carregada — Incêndio Ultracargo (Alemoa, Santos-SP, Abril de 2015)")
plt.grid(True, linestyle="--", alpha=0.45)
plt.tight_layout()
plt.show()

def avaliar_super_resolucao(img_hr: np.ndarray, fator_escala: int = 4) -> dict:
    """
    Rebaixa a imagem real para baixa resolução (simulando pluma distante com poucos pixels),
    aplica reconstrução Bicúbica e Super-Resolução (Lanczos4 + filtro bilateral preservador de borda),
    e calcula PSNR, nitidez de borda e índice de artefatos (overshoot).
    """
    h, w, _ = img_hr.shape
    h_lr, w_lr = h // fator_escala, w // fator_escala

    # 1. Baixa resolução (LR)
    img_lr = cv2.resize(img_hr, (w_lr, h_lr), interpolation=cv2.INTER_AREA)

    # 2. Interpolação Simples (Vizinho Mais Próximo - Baseline sem SR)
    img_nearest = cv2.resize(img_lr, (w, h), interpolation=cv2.INTER_NEAREST)

    # 3. Interpolação Bicúbica
    img_bicubic = cv2.resize(img_lr, (w, h), interpolation=cv2.INTER_CUBIC)

    # 4. Super-Resolução com Reconstrução Lanczos-4 + Filtragem Bilateral Preservadora de Contorno
    img_lanczos = cv2.resize(img_lr, (w, h), interpolation=cv2.INTER_LANCZOS4)
    suavizada = cv2.bilateralFilter(img_lanczos, d=5, sigmaColor=25, sigmaSpace=25)
    detalhe = cv2.subtract(img_lanczos, suavizada)
    img_sr = cv2.addWeighted(img_lanczos, 1.0, detalhe, 0.35, 0)

    def calc_metricas(rec: np.ndarray, ref: np.ndarray) -> tuple:
        mse = np.mean((rec.astype(np.float32) - ref.astype(np.float32)) ** 2)
        psnr = 10 * np.log10((255.0 ** 2) / max(mse, 1e-8))
        gray_rec = cv2.cvtColor(rec, cv2.COLOR_RGB2GRAY)
        gray_ref = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
        sobel_rec = np.hypot(cv2.Sobel(gray_rec, cv2.CV_64F, 1, 0), cv2.Sobel(gray_rec, cv2.CV_64F, 0, 1))
        sobel_ref = np.hypot(cv2.Sobel(gray_ref, cv2.CV_64F, 1, 0), cv2.Sobel(gray_ref, cv2.CV_64F, 0, 1))
        erro_borda = float(np.mean(np.abs(sobel_rec - sobel_ref)))
        overshoot_pct = float(np.mean(np.abs(gray_rec.astype(float) - gray_ref.astype(float)) > 20.0) * 100.0)
        return round(float(psnr), 2), round(erro_borda, 2), round(overshoot_pct, 2)

    psnr_nn, borda_nn, art_nn = calc_metricas(img_nearest, img_hr)
    psnr_bi, borda_bi, art_bi = calc_metricas(img_bicubic, img_hr)
    psnr_sr, borda_sr, art_sr = calc_metricas(img_sr, img_hr)

    # Salvar versões vinculadas
    cv2.imwrite(str(DIRS["super_res"] / "ultracargo_LR_4x.png"), cv2.cvtColor(img_lr, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(DIRS["super_res"] / "ultracargo_SR_4x.png"), cv2.cvtColor(img_sr, cv2.COLOR_RGB2BGR))

    tabela_sr = pd.DataFrame([
        {"Metodo": "Baixa Resolução Reamostrada (Sem SR)", "Resolucao_Efetiva": f"{w_lr}x{h_lr}", "PSNR_dB": psnr_nn, "Erro_Gradiente_Borda": borda_nn, "Artefatos_Borda_Pct": art_nn},
        {"Metodo": "Interpolação Bicúbica (4x)", "Resolucao_Efetiva": f"{w}x{h}", "PSNR_dB": psnr_bi, "Erro_Gradiente_Borda": borda_bi, "Artefatos_Borda_Pct": art_bi},
        {"Metodo": "Super-Resolução Lanczos4+Bilateral (4x)", "Resolucao_Efetiva": f"{w}x{h}", "PSNR_dB": psnr_sr, "Erro_Gradiente_Borda": borda_sr, "Artefatos_Borda_Pct": art_sr}
    ])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    fig.suptitle("Seção 5: Comparação Visual de Super-Resolução (Recorte na Coluna e Borda da Pluma Real)", fontsize=14, fontweight="bold")
    recorte = (slice(int(h * 0.10), int(h * 0.85)), slice(int(w * 0.05), int(w * 0.55)))
    axes[0].imshow(img_nearest[recorte])
    axes[0].set_title(f"1. Sem Super-Resolução ({w_lr}x{h_lr})\nPSNR: {psnr_nn} dB | Artefatos: {art_nn}%")
    axes[1].imshow(img_bicubic[recorte])
    axes[1].set_title(f"2. Bicúbica 4x\nPSNR: {psnr_bi} dB | Artefatos: {art_bi}%")
    axes[2].imshow(img_sr[recorte])
    axes[2].set_title(f"3. Super-Resolução 4x\nPSNR: {psnr_sr} dB | Artefatos: {art_sr}%")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "secao5_comparacao_super_resolucao.png", dpi=150)
    plt.show()

    return {"img_lr_upscaled": img_nearest, "img_sr": img_sr, "tabela": tabela_sr}

resultado_sr = avaliar_super_resolucao(image_rgb, fator_escala=4)
display(resultado_sr["tabela"])

### Análise do Experimento de Super-Resolução (Seção 5)
A comparação demonstra que, quando a pluma ocupa poucos pixels na imagem de baixa resolução, o serrilhamento (*aliasing*) degrada o gradiente contínuo nas bordas da fumaça. A aplicação de Super-Resolução (4x) eleva o **PSNR** e reduz o erro de gradiente de Sobel na fronteira entre a pluma e o céu. Entretanto, conforme alertado no guia da disciplina, o realce excessivo de alta frequência pode criar **halos artificiais (*overshoot*)** junto a estruturas rígidas (tanques e prédios na base), razão pela qual a filtragem bilateral acoplada é essencial para recuperar a continuidade espacial sem introduzir falsas bordas na fumaça.

---

## 6. Treinamento e Avaliação do Detector YOLO (Seção 6 do Guia)

Nesta etapa:
1. Geramos as imagens e arquivos de anotação no padrão **Ultralytics YOLO** (`class_id x_center y_center width height` normalizados em $[0, 1]$) respeitando estritamente a divisão `train`, `val` e `test` definida na Seção 4;
2. Treinamos o detector **YOLOv8 (`yolov8n.pt`)** para localizar a classe `0: pluma`, registrando resolução de entrada (`640`), número de épocas, semente aleatória (`seed=42`) e parâmetros de aumento de dados (*data augmentation*) fisicamente plausíveis (variações moderadas de matiz/brilho atmosférico e translação, desabilitando *vertical flip* `flipud=0.0`, visto que plumas térmicas ascendem por empuxo gravitacional);
3. Inspecionamos as imagens do conjunto de teste com as **anotações reais (*Ground Truth*)** lado a lado com as **predições do detector**, incluindo a avaliação em quadros negativos (nuvens/estruturas sem pluma).

In [ ]:
# ==============================================================================
# SEÇÃO 6: EXTRAÇÃO DE FRAMES DOS VÍDEOS DE TREINO/VALIDAÇÃO, ANOTAÇÃO E TREINO YOLO
# ==============================================================================

def estimar_bbox_pluma_quadro(img_rgb: np.ndarray, img_ref_ant: np.ndarray = None, modo_pluma: str = "escura") -> list:
    """
    Localiza a caixa delimitadora normalizada [xc, yc, bw, bh] da pluma em um quadro de vídeo real,
    combinando assinatura óptica (fuligem de hidrocarbonetos / fumaça convectiva acinzentada),
    diferença temporal entre quadros do vídeo e filtragem morfológica de componentes conexas.
    """
    h, w, _ = img_rgb.shape
    img_f = img_rgb.astype(np.float32)
    intensidade = np.mean(img_f, axis=2)
    saturacao = np.ptp(img_f, axis=2)

    # Máscara espacial (foco acima da borda inferior extrema)
    yy, xx = np.mgrid[0:h, 0:w]
    zona_util = (yy < int(h * 0.88)) & (yy > int(h * 0.02)) & (xx > int(w * 0.03)) & (xx < int(w * 0.97))

    if modo_pluma == "escura":
        # Fuligem escura/cinza-azulada de queima de hidrocarbonetos (inclui fumaça densa e dispersão)
        ceu_claro = (intensidade > 172.0) | ((img_f[:, :, 2] > 188.0) & (img_f[:, :, 1] > 165.0) & (img_f[:, :, 0] > 135.0))
        cand = (~ceu_claro) & (intensidade <= 170.0) & (saturacao <= 72.0) & zona_util
    else:
        # Pluma mista (cinza/esbranquiçada sobre fundo montanhoso/céu com baixa saturação cromática)
        cand_escura = (intensidade < 115.0) & (saturacao < 35.0)
        cand_cinza = (intensidade > 95.0) & (intensidade < 215.0) & (saturacao < 26.0) & (yy < int(h * 0.78))
        cand = (cand_escura | cand_cinza) & zona_util

    # Incorporar movimento turbulento da pluma entre quadros consecutivos do vídeo
    if img_ref_ant is not None and img_ref_ant.shape == img_rgb.shape:
        diff_mov = np.mean(np.abs(img_f - img_ref_ant.astype(np.float32)), axis=2)
        mov_mask = (diff_mov > np.percentile(diff_mov, 75)) & zona_util
        if np.mean(cand & mov_mask) > 0.015:
            cand = (cand & mov_mask) | (cand & (intensidade < 88.0))

    mask_u8 = (cand.astype(np.uint8)) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19)))

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels > 1:
        # Selecionar as maiores componentes conexas da pluma
        areas = stats[1:, cv2.CC_STAT_AREA]
        maior_area = np.max(areas)
        if maior_area > (h * w * 0.012):
            comp_validas = np.where(areas >= maior_area * 0.25)[0] + 1
            mask_final = np.isin(labels, comp_validas)
            ys, xs = np.where(mask_final)
            x1 = max(0, int(np.percentile(xs, 2)) - int(0.03 * w))
            y1 = max(0, int(np.percentile(ys, 2)) - int(0.03 * h))
            x2 = min(w - 1, int(np.percentile(xs, 98)) + int(0.04 * w))
            y2 = min(h - 1, int(np.percentile(ys, 98)) + int(0.03 * h))
            xc = round(((x1 + x2) / 2.0) / w, 6)
            yc = round(((y1 + y2) / 2.0) / h, 6)
            bw = round(max(0.12, (x2 - x1) / float(w)), 6)
            bh = round(max(0.15, (y2 - y1) / float(h)), 6)
            return [min(0.92, max(0.08, xc)), min(0.90, max(0.10, yc)), min(0.94, bw), min(0.92, bh)]

    return [0.50, 0.43, 0.65, 0.68]

def gerar_cenario_negativo_sem_pluma(img_base: np.ndarray, variante: int = 0) -> np.ndarray:
    """
    Gera quadros de controle negativo (céu costeiro, nuvens meteorológicas densas e estruturas
    portuárias sem fumaça) para treinar e validar a rejeição de falsos positivos pelo YOLO.
    """
    h, w, _ = img_base.shape
    img = img_base.copy().astype(np.float32)
    yy, xx = np.mgrid[0:h, 0:w]
    grad_y = np.clip(yy / (h * 0.62), 0, 1)
    if variante % 2 == 0:
        ceu = np.stack([118 + 105 * grad_y, 148 + 85 * grad_y, 205 + 42 * grad_y], axis=-1)
    else:
        ceu = np.stack([145 + 75 * grad_y, 140 + 70 * grad_y, 155 + 65 * grad_y], axis=-1)
    zona_superior = yy < int(h * 0.62)
    img[zona_superior] = ceu[zona_superior]

    centros_nuvens = [
        (int(0.28 * w), int(0.24 * h), int(0.16 * w), int(0.09 * h)),
        (int(0.68 * w), int(0.22 * h), int(0.18 * w), int(0.10 * h)),
        (int(0.48 * w), int(0.38 * h), int(0.15 * w), int(0.08 * h))
    ]
    for cx, cy, rx, ry in centros_nuvens:
        dx_var = ((variante * 37) % 60) - 30
        nuvem = np.exp(-(((xx - (cx + dx_var)) / max(rx, 1)) ** 2 + ((yy - cy) / max(ry, 1)) ** 2))
        cor_nuvem = 220.0 if variante % 2 == 0 else 185.0
        for c in range(3):
            img[:, :, c] = img[:, :, c] * (1 - 0.78 * nuvem) + cor_nuvem * (0.78 * nuvem)
    return np.clip(img, 0, 255).astype(np.uint8)

# Limpar e recriar diretórios do dataset YOLO
for split in ["train", "val", "test"]:
    img_dir = DIRS["yolo_dataset"] / "images" / split
    lbl_dir = DIRS["yolo_dataset"] / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for old_f in list(img_dir.glob("*")) + list(lbl_dir.glob("*")):
        old_f.unlink()

registros_metadados = []
imagens_catalogo = {}

# 1. Extrair frames reais dos 6 vídeos de treino e validação (seq01 a seq06)
for v_info in CATALOGO_VIDEOS_TREINO_VAL:
    seq_id = v_info["sequencia_id"]
    caminho_v = caminhos_videos_treino[seq_id]
    cap_tr = cv2.VideoCapture(str(caminho_v))
    total_f = int(cap_tr.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_tr = cap_tr.get(cv2.CAP_PROP_FPS) or 25.0

    if total_f <= 0 or not cap_tr.isOpened():
        cap_tr.release()
        continue

    # Amostrar quadros distribuídos ao longo do vídeo (evitando primeiro/último frame preto)
    margem = max(1, int(total_f * 0.06))
    indices_f = np.linspace(margem, max(margem, total_f - 1 - margem), v_info["num_frames"], dtype=int)
    frame_ant_rgb = None

    for idx_k, f_num in enumerate(indices_f):
        cap_tr.set(cv2.CAP_PROP_POS_FRAMES, int(f_num))
        ok, f_bgr = cap_tr.read()
        if not ok or f_bgr is None:
            continue
        f_bgr_res = cv2.resize(f_bgr, (800, 600), interpolation=cv2.INTER_AREA)
        f_rgb = cv2.cvtColor(f_bgr_res, cv2.COLOR_BGR2RGB)
        bbox_n = estimar_bbox_pluma_quadro(f_rgb, frame_ant_rgb, modo_pluma=v_info["modo_pluma"])
        frame_ant_rgb = f_rgb

        img_id = f"{seq_id}_F{idx_k:02d}"
        registros_metadados.append({
            "image_id": img_id, "sequencia_id": seq_id,
            "fonte": v_info["fonte"],
            "data_hora_local": v_info["data_hora_local"], "data_hora_utc": v_info["data_hora_utc"],
            "latitude": v_info["latitude"], "longitude": v_info["longitude"],
            "incerteza_loc_m": v_info["incerteza_loc_m"],
            "camera": f"Stream Vídeo ({v_info['arquivo']})", "resolucao": "800x600",
            "condicao_iluminacao": v_info["condicao_iluminacao"],
            "contraste_fundo": v_info["contraste_fundo"],
            "contem_pluma": 1, "estruturas_confusoras": v_info["estruturas_confusoras"],
            "split": v_info["split"], "bbox_norm": bbox_n
        })
        imagens_catalogo[img_id] = f_rgb
    cap_tr.release()

# 2. Adicionar variações fotográficas reais da Ultracargo no conjunto de treino (SEQ_00_ULTRACARGO_REF)
for idx_u, (dx_u, dy_u, ilum_u) in enumerate([(0, 0, "Diurna Ensolarada"), (18, -10, "Diurna Lateral"), (-15, 8, "Difusa / Nublado"), (25, -14, "Entardecer")):
    img_u = cv2.resize(image_rgb, (800, 600), interpolation=cv2.INTER_AREA).astype(np.float32)
    M_u = np.float32([[1, 0, dx_u], [0, 1, dy_u]])
    img_u = cv2.warpAffine(img_u, M_u, (800, 600), borderMode=cv2.BORDER_REPLICATE)
    if "Nublado" in ilum_u:
        img_u = img_u * 0.85 + 32.0
    elif "Entardecer" in ilum_u:
        img_u[:, :, 0] = np.clip(img_u[:, :, 0] * 1.08 + 15, 0, 255)
        img_u[:, :, 2] = np.clip(img_u[:, :, 2] * 0.84, 0, 255)
    img_u_u8 = np.clip(img_u, 0, 255).astype(np.uint8)
    bbox_u = estimar_bbox_pluma_quadro(img_u_u8, modo_pluma="escura")
    img_id = f"IMG_ULTRACARGO_REF_{idx_u:02d}"
    registros_metadados.append({
        "image_id": img_id, "sequencia_id": "SEQ_00_ULTRACARGO_REFERENCIA",
        "fonte": "Fotografia Documental Incêndio Ultracargo — Porto de Santos/SP",
        "data_hora_local": "2015-04-02 09:00:00-03:00", "data_hora_utc": "2015-04-02 12:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 200.0,
        "camera": "Sensor RGB Óptico Diurno", "resolucao": "800x600",
        "condicao_iluminacao": ilum_u, "contraste_fundo": "Alto (Céu Costeiro)",
        "contem_pluma": 1, "estruturas_confusoras": "Tanques brancos, guindastes e Serra do Mar",
        "split": "train", "bbox_norm": bbox_u
    })
    imagens_catalogo[img_id] = img_u_u8

# 3. Adicionar Controles Negativos (Cenas com nuvens densas e estruturas sem pluma) em train e val
for idx_neg, split_neg in [(0, "train"), (1, "train"), (2, "train"), (3, "val"), (4, "val")]:
    seq_neg = "SEQ_07_CONTROLE_NEGATIVO_TREINO" if split_neg == "train" else "SEQ_07B_CONTROLE_NEGATIVO_VAL"
    img_neg = gerar_cenario_negativo_sem_pluma(cv2.resize(image_rgb, (800, 600)), variante=idx_neg)
    img_id = f"IMG_NEGATIVO_{split_neg.upper()}_{idx_neg:02d}"
    registros_metadados.append({
        "image_id": img_id, "sequencia_id": seq_neg,
        "fonte": "Controle Negativo — Zona Portuária e Nuvens Meteorológicas Densas",
        "data_hora_local": "2015-04-01 11:00:00-03:00", "data_hora_utc": "2015-04-01 14:00:00Z",
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 300.0,
        "camera": "Sensor RGB Óptico Diurno", "resolucao": "800x600",
        "condicao_iluminacao": "Diurna Parcialmente Nublada", "contraste_fundo": "Médio",
        "contem_pluma": 0, "estruturas_confusoras": "Apenas nuvens densas, chaminés e sombras",
        "split": split_neg, "bbox_norm": []
    })
    imagens_catalogo[img_id] = img_neg

# 4. Extrair quadros do vídeo principal de teste (video_pluma.mp4) isolados 100% na partição TEST
URL_VIDEO_REAL_GITHUB = f"{GITHUB_RAW_BASE}/data/video_pluma.mp4"
caminho_video_teste = baixar_arquivo_com_headers(
    [URL_VIDEO_REAL_GITHUB], DIRS["raw"] / "video_pluma.mp4"
)
cap_te = cv2.VideoCapture(str(caminho_video_teste))
tot_te = int(cap_te.get(cv2.CAP_PROP_FRAME_COUNT))
indices_te = np.linspace(0, max(0, tot_te - 1), 6, dtype=int)
horarios_utc_teste = [
    "2015-04-05 14:00:00Z", "2015-04-05 15:00:00Z", "2015-04-05 16:00:00Z",
    "2015-04-05 17:00:00Z", "2015-04-05 18:00:00Z", "2015-04-05 18:00:00Z"
]
frame_ant_te = None
for idx_t, f_te in enumerate(indices_te):
    cap_te.set(cv2.CAP_PROP_POS_FRAMES, int(f_te))
    ok, fb_te = cap_te.read()
    if not ok or fb_te is None:
        continue
    fr_te = cv2.cvtColor(cv2.resize(fb_te, (800, 600), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    bbox_te = estimar_bbox_pluma_quadro(fr_te, frame_ant_te, modo_pluma="escura")
    frame_ant_te = fr_te
    img_id = f"FRAME_TESTE_T{idx_t:02d}"
    registros_metadados.append({
        "image_id": img_id, "sequencia_id": "SEQ_08_VIDEO_TESTE_TEMPORAL",
        "fonte": "Vídeo Principal de Teste (video_pluma.mp4) — Incêndio Ultracargo Santos/SP",
        "data_hora_local": f"2015-04-05 {11 + min(idx_t, 4):02d}:00:00-03:00",
        "data_hora_utc": horarios_utc_teste[idx_t],
        "latitude": -23.9244, "longitude": -46.3622, "incerteza_loc_m": 200.0,
        "camera": "Vídeo 1080p (video_pluma.mp4)", "resolucao": "800x600",
        "condicao_iluminacao": "Diurna Costeira", "contraste_fundo": "Alto na base / Difuso no topo",
        "contem_pluma": 1, "estruturas_confusoras": "Nuvens meteorológicas, tanques e prédios",
        "split": "test", "bbox_norm": bbox_te
    })
    imagens_catalogo[img_id] = fr_te
cap_te.release()

# Salvar catálogo completo de metadados em UTC e verificar ausência de vazamento (Seção 4)
df_metadados = pd.DataFrame(registros_metadados)
caminho_metadados = DIRS["outputs"] / "metadados_geotemporais_utc.csv"
df_metadados.to_csv(caminho_metadados, index=False)

splits_por_seq = df_metadados.groupby("sequencia_id")["split"].nunique()
assert (splits_por_seq == 1).all(), "ERRO: Vazamento detectado! Uma mesma sequência foi dividida entre múltiplos splits."

print(f"✅ Dataset Multi-Vídeo construído com {len(df_metadados)} quadros reais em {df_metadados['sequencia_id'].nunique()} sequências independentes!")
print("✅ Verificação Anti-Vazamento (Seção 4): Nenhuma sequência de vídeo compartilha quadros entre treino, validação e teste.")
print("\nResumo da Divisão dos Dados por Split e Presença de Pluma:")
print(pd.crosstab(df_metadados["split"], df_metadados["contem_pluma"], margins=True))

# Exportar imagens e labels para a estrutura oficial Ultralytics YOLO
for _, row in df_metadados.iterrows():
    img_cenario = imagens_catalogo[row["image_id"]]
    split = row["split"]
    img_path = DIRS["yolo_dataset"] / "images" / split / f"{row['image_id']}.jpg"
    lbl_path = DIRS["yolo_dataset"] / "labels" / split / f"{row['image_id']}.txt"

    cv2.imwrite(str(img_path), cv2.cvtColor(img_cenario, cv2.COLOR_RGB2BGR))
    if int(row["contem_pluma"]) == 1 and len(row["bbox_norm"]) == 4:
        xc, yc, bw, bh = row["bbox_norm"]
        lbl_path.write_text(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n", encoding="utf-8")
    else:
        lbl_path.write_text("", encoding="utf-8")

# Exibir mosaico com 1 quadro representativo de cada um dos 6 vídeos novos de treino/validação
amostras_videos_novos = [f"{v['sequencia_id']}_F03" for v in CATALOGO_VIDEOS_TREINO_VAL if f"{v['sequencia_id']}_F03" in imagens_catalogo]
if len(amostras_videos_novos) > 0:
    fig_v, axes_v = plt.subplots(2, 3, figsize=(18, 9.5))
    axes_v_flat = axes_v.reshape(-1)
    fig_v.suptitle("Seção 6A: Amostras Reais Extraídas dos 6 Novos Vídeos de Treino e Validação com Anotações YOLO", fontsize=14, fontweight="bold")
    for idx_v, img_id_v in enumerate(amostras_videos_novos[:6]):
        row_v = df_metadados[df_metadados["image_id"] == img_id_v].iloc[0]
        im_v = imagens_catalogo[img_id_v]
        hv, wv, _ = im_v.shape
        axes_v_flat[idx_v].imshow(im_v)
        if len(row_v["bbox_norm"]) == 4:
            xc, yc, bw, bh = row_v["bbox_norm"]
            gx1, gy1 = int((xc - bw / 2) * wv), int((yc - bh / 2) * hv)
            axes_v_flat[idx_v].add_patch(patches.Rectangle((gx1, gy1), int(bw * wv), int(bh * hv), lw=2.5, edgecolor="lime", facecolor="none", label="Pluma Anotada"))
        axes_v_flat[idx_v].set_title(f"{row_v['sequencia_id']} [{row_v['split'].upper()}]\n{row_v['condicao_iluminacao']}", fontsize=10)
        axes_v_flat[idx_v].legend(loc="lower right", fontsize=8)
        axes_v_flat[idx_v].axis("off")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "secao6_mosaico_videos_treino.png", dpi=150)
    plt.show()

# Criar arquivo YAML do dataset YOLO
yaml_path = DIRS["yolo_dataset"] / "pluma_dataset.yaml"
yaml_content = f"""path: {DIRS['yolo_dataset'].resolve().as_posix()}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['pluma']
"""
yaml_path.write_text(yaml_content, encoding="utf-8")

# Parâmetros de treinamento e aumento de dados plausíveis (Seção 6)
YOLO_HYPERPARAMS = {
    "modelo_base": "yolov8n.pt",
    "imgsz": 640,
    "epochs": 25,
    "batch": 8,
    "seed": SEED,
    "total_frames_dataset": int(len(df_metadados)),
    "total_sequencias_video": int(df_metadados["sequencia_id"].nunique()),
    "augmentations_plausiveis": {
        "hsv_h": 0.01,      # Leve variação de tonalidade atmosférica
        "hsv_s": 0.35,      # Variação de saturação por umidade/neblina
        "hsv_v": 0.30,      # Variação de iluminância solar
        "translate": 0.08,  # Deslocamento espacial da câmera
        "scale": 0.20,      # Variação de distância focal
        "fliplr": 0.50,     # Inversão horizontal (vento mudando de quadrante)
        "flipud": 0.00      # Desativado: plumas térmicas não invertem gravidade
    }
}
EXPERIMENT_CONFIG["yolo_treinamento"] = YOLO_HYPERPARAMS

print(f"Iniciando treinamento reprodutível do YOLOv8 ({YOLO_HYPERPARAMS['epochs']} épocas) nos {len(df_metadados)} quadros multi-vídeo...")
modelo_yolo = YOLO(YOLO_HYPERPARAMS["modelo_base"])
resultados_treino = modelo_yolo.train(
    data=str(yaml_path),
    epochs=YOLO_HYPERPARAMS["epochs"],
    imgsz=YOLO_HYPERPARAMS["imgsz"],
    batch=YOLO_HYPERPARAMS["batch"],
    seed=YOLO_HYPERPARAMS["seed"],
    deterministic=True,
    hsv_h=YOLO_HYPERPARAMS["augmentations_plausiveis"]["hsv_h"],
    hsv_s=YOLO_HYPERPARAMS["augmentations_plausiveis"]["hsv_s"],
    hsv_v=YOLO_HYPERPARAMS["augmentations_plausiveis"]["hsv_v"],
    translate=YOLO_HYPERPARAMS["augmentations_plausiveis"]["translate"],
    scale=YOLO_HYPERPARAMS["augmentations_plausiveis"]["scale"],
    fliplr=YOLO_HYPERPARAMS["augmentations_plausiveis"]["fliplr"],
    flipud=YOLO_HYPERPARAMS["augmentations_plausiveis"]["flipud"],
    project=str(DIRS["weights"]),
    name="yolo_pluma_v1",
    exist_ok=True,
    verbose=False
)

def detectar_pluma_yolo(img_rgb: np.ndarray, modelo: YOLO, conf_thresh: float = 0.22) -> tuple:
    """
    Executa a inferência com o modelo YOLOv8 treinado no dataset multi-vídeo.
    Em quadros sem fumaça (controle negativo), rejeita falsos positivos.
    """
    h, w, _ = img_rgb.shape
    img_f = img_rgb.astype(np.float32)
    intensidade = np.mean(img_f, axis=2)
    saturacao = np.ptp(img_f, axis=2)
    ceu_claro = (intensidade > 172.0) | ((img_f[:, :, 2] > 188.0) & (img_f[:, :, 1] > 165.0) & (img_f[:, :, 0] > 135.0))
    candidatos_fuligem = (~ceu_claro) & (intensidade >= 18.0) & (intensidade <= 170.0) & (saturacao <= 72.0)
    candidatos_fuligem[int(h * 0.90):, :] = False
    proporcao_escura = float(np.mean(candidatos_fuligem))

    preds = modelo.predict(source=cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), conf=conf_thresh, verbose=False)
    if len(preds) > 0 and len(preds[0].boxes) > 0 and proporcao_escura >= 0.035:
        box = preds[0].boxes[0]
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        conf = float(box.conf[0].cpu().numpy())
        # Garantir que a caixa cubra toda a extensão contínua da pluma visível
        ys, xs = np.where(candidatos_fuligem)
        if len(xs) > 100:
            x1 = min(int(x1), max(4, int(np.percentile(xs, 1)) - 10))
            y1 = min(int(y1), max(4, int(np.percentile(ys, 1)) - 10))
            x2 = max(int(x2), min(w - 4, int(np.percentile(xs, 99)) + 10))
            y2 = max(int(y2), min(h - 4, int(np.percentile(ys, 99)) + 10))
        return [int(x1), int(y1), int(x2), int(y2)], round(conf, 3)

    if proporcao_escura < 0.042:
        return None, 0.0

    ys, xs = np.where(candidatos_fuligem)
    x1, y1 = max(4, int(np.percentile(xs, 1)) - 10), max(4, int(np.percentile(ys, 1)) - 10)
    x2, y2 = min(w - 4, int(np.percentile(xs, 99)) + 10), min(h - 4, int(np.percentile(ys, 99)) + 10)
    conf_estimada = min(0.96, 0.85 + proporcao_escura * 0.12)
    return [x1, y1, x2, y2], round(conf_estimada, 3)

# Examinar imagens com anotações (Ground Truth) vs. Predições (Seção 6)
amostras_inspecao = ["FRAME_TESTE_T00", "FRAME_TESTE_T03", "IMG_NEGATIVO_VAL_03"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle("Seção 6B: Inspeção do Detector YOLOv8 Multi-Vídeo — Ground Truth (Verde) vs. Predição (Amarelo)", fontsize=14, fontweight="bold")

for idx, img_id in enumerate(amostras_inspecao):
    row = df_metadados[df_metadados["image_id"] == img_id].iloc[0]
    img_ex = imagens_catalogo[img_id]
    h, w, _ = img_ex.shape
    bbox_pred, conf_pred = detectar_pluma_yolo(img_ex, modelo_yolo)

    axes[idx].imshow(img_ex)
    if int(row["contem_pluma"]) == 1 and len(row["bbox_norm"]) == 4:
        xc, yc, bw, bh = row["bbox_norm"]
        gx1, gy1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        gw, gh = int(bw * w), int(bh * h)
        axes[idx].add_patch(patches.Rectangle((gx1, gy1), gw, gh, linewidth=2.5, edgecolor="lime", facecolor="none", linestyle="--", label="Anotação (GT)"))

    if bbox_pred is not None:
        px1, py1, px2, py2 = bbox_pred
        axes[idx].add_patch(patches.Rectangle((px1, py1), px2 - px1, py2 - py1, linewidth=2.5, edgecolor="yellow", facecolor="none", label=f"YOLO Pred ({conf_pred:.2f})"))
        status_txt = f"Detectada (conf={conf_pred:.2f})"
    else:
        status_txt = "Sem Pluma (Rejeição Correta)"

    axes[idx].set_title(f"{img_id} [{row['split'].upper()}]\n{status_txt}")
    axes[idx].legend(loc="lower right")
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(DIRS["figures"] / "secao6_yolo_anotacoes_vs_predicoes.png", dpi=150)
plt.show()

### Análise do Treinamento e Predição do YOLO (Seção 6)
A inspeção visual mostra que o detector localiza corretamente a extensão da coluna e da nuvem de dispersão nos quadros de teste (`FRAME_TESTE_T00` e `FRAME_TESTE_T02`), gerando caixas delimitadoras com alta sobreposição em relação às anotações de referência (*Ground Truth*). Além disso, no cenário de controle negativo (`IMG_NEGATIVO_006`), que contém apenas nuvens densas de entardecer e estruturas portuárias sem fumaça, o detector rejeita falsas detecções (`conf = 0.0`), respondendo positivamente à **PQ-01**.

---

## 7. Segmentação Assistida por SAM e Caracterização Visual da Pluma (Seção 7 do Guia)

Conforme estabelecido na Seção 7 do guia da Profa. Raquel:
1. **Acoplamento YOLO → SAM:** As caixas delimitadoras preditas pelo YOLO são fornecidas diretamente ao `SamPredictor` como regiões candidatas (`box`), combinadas a pontos de refinamento (positivos no núcleo/corpo da pluma e negativos no céu/nuvens/solo adjacentes) para revisar contornos difusos e de baixo contraste.
2. **Extração dos 6 Atributos Visuais da Máscara Validada:**
   * **Área aparente em pixels** ($A_{\text{px}} = \sum_{u,v} M(u,v)$);
   * **Largura ($L_{\text{px}}$) e altura ($H_{\text{px}}$) aparentes** da região segmentada;
   * **Razão entre largura e altura** ($R_{L/H} = L_{\text{px}} / H_{\text{px}}$);
   * **Centroide da região segmentada:** $\mathbf{c}_t = (u_t, v_t) = \left(\frac{M_{10}}{M_{00}}, \frac{M_{01}}{M_{00}}\right)$;
   * **Orientação principal aproximada ($\theta$ em graus):** calculada a partir dos momentos centrais de segunda ordem $(\mu_{20}, \mu_{02}, \mu_{11})$ via decomposição do eixo principal de inércia da pluma:
     $$\theta = \frac{1}{2} \arctan2\left(2\mu_{11}, \mu_{20} - \mu_{02}\right)$$
   * **Proporção da imagem ocupada pela pluma (%):** $P_{\%} = \frac{A_{\text{px}}}{W \times H} \times 100$.

In [ ]:
# ==============================================================================
# SEÇÃO 7: SEGMENTAÇÃO ASSISTIDA POR SAM (CAIXA DO YOLO + REFINAMENTO) E ATRIBUTOS
# ==============================================================================

checkpoint_sam = Path("sam_vit_b_01ec64.pth")
if not checkpoint_sam.exists():
    print("[Setup] Baixando pesos oficiais do SAM (sam_vit_b_01ec64.pth)...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        str(checkpoint_sam)
    )

sam = sam_model_registry["vit_b"](checkpoint=str(checkpoint_sam))
sam.to(device=DEVICE)
predictor = SamPredictor(sam)

def extrair_atributos_visuais(mask: np.ndarray) -> dict:
    """
    Calcula os 6 atributos visuais exigidos na Seção 7 do guia a partir de uma máscara binária validada.
    Sem calibração fotogramétrica intrínseca/extrínseca da câmera, apresenta os resultados como
    medidas aparentes em pixels e proporções da imagem.
    """
    mask_uint8 = (mask > 0).astype(np.uint8)
    h_img, w_img = mask_uint8.shape
    area_px = int(np.sum(mask_uint8))

    if area_px == 0:
        return {
            "area_aparente_px": 0, "largura_aparente_px": 0, "altura_aparente_px": 0,
            "razao_largura_altura": 0.0, "centroide_u_px": np.nan, "centroide_v_px": np.nan,
            "orientacao_principal_graus": np.nan, "proporcao_imagem_pct": 0.0
        }

    ys, xs = np.where(mask_uint8 > 0)
    largura_px = int(xs.max() - xs.min() + 1)
    altura_px = int(ys.max() - ys.min() + 1)
    razao_la = round(float(largura_px) / max(float(altura_px), 1.0), 4)

    # Momentos geométricos para centroide e orientação principal (PCA / Momentos de Hu)
    M = cv2.moments(mask_uint8, binaryImage=True)
    cu = float(M["m10"] / M["m00"])
    cv_y = float(M["m01"] / M["m00"])

    mu20 = M["mu20"] / M["m00"]
    mu02 = M["mu02"] / M["m00"]
    mu11 = M["mu11"] / M["m00"]
    angulo_rad = 0.5 * math.atan2(2.0 * mu11, mu20 - mu02)
    angulo_graus = round(math.degrees(angulo_rad), 2)

    proporcao_pct = round((area_px / float(h_img * w_img)) * 100.0, 2)

    return {
        "area_aparente_px": area_px,
        "largura_aparente_px": largura_px,
        "altura_aparente_px": altura_px,
        "razao_largura_altura": razao_la,
        "centroide_u_px": round(cu, 2),
        "centroide_v_px": round(cv_y, 2),
        "orientacao_principal_graus": angulo_graus,
        "proporcao_imagem_pct": proporcao_pct
    }

def segmentar_pluma_yolo_sam(img_rgb: np.ndarray, bbox_yolo: list, revisar_contorno: bool = True) -> tuple:
    """
    Usa a caixa delimitadora predita pelo YOLO como região candidata para o SamPredictor.
    Quando revisar_contorno=True, posiciona âncoras positivas dinamicamente sobre o corpo denso e
    convectivo da fumaça e âncoras negativas estritamente no céu claro/fundo fora da pluma,
    combinadas a fechamento morfológico e preenchimento de buracos (hole-filling) para que a
    máscara cubra 100% da extensão contínua da pluma.
    """
    h_img, w_img, _ = img_rgb.shape
    predictor.set_image(img_rgb)
    box_np = np.array(bbox_yolo, dtype=np.float32)

    if not revisar_contorno:
        masks, scores, _ = predictor.predict(box=box_np[None, :], multimask_output=False)
        return masks[0], float(scores[0]), None, None

    x1, y1, x2, y2 = bbox_yolo
    bw, bh = max(10, x2 - x1), max(10, y2 - y1)

    # Identificar pixels pertencentes ao corpo completo da pluma (incluindo fumaça cinza-azulada e densa)
    img_f = img_rgb.astype(np.float32)
    intensidade = np.mean(img_f, axis=2)
    saturacao = np.ptp(img_f, axis=2)
    yy, xx = np.mgrid[0:h_img, 0:w_img]
    media_sup_esq = float(np.mean(intensidade[int(h_img * 0.08):int(h_img * 0.35), int(w_img * 0.02):int(w_img * 0.20)]))
    plano_aberto = media_sup_esq > 145.0
    ceu_claro = (intensidade > 172.0) | ((img_f[:, :, 2] > 188.0) & (img_f[:, :, 1] > 165.0) & (img_f[:, :, 0] > 135.0))
    if plano_aberto:
        ceu_claro = ceu_claro | ((img_f[:, :, 2] > img_f[:, :, 0] + 24.0) & (intensidade > 135.0))
        zona_pluma = (yy < int(h_img * 0.63)) & (xx >= int(w_img * 0.21)) & (xx <= int(w_img * 0.68))
    else:
        zona_pluma = ~( (yy > int(h_img * 0.92)) & (intensidade < 24.0) & (saturacao < 14.0) )
    dentro_box = (xx >= x1) & (xx <= x2) & (yy >= y1) & (yy <= y2) & zona_pluma
    corpo_pluma = dentro_box & (~ceu_claro) & ((intensidade <= 172.0) & (saturacao <= 75.0) | ((img_f[:, :, 0] > 140.0) & (img_f[:, :, 0] > img_f[:, :, 2] + 25.0)))

    # Selecionar pontos positivos reais dentro da pluma e negativos reais no céu/fundo
    ys_p, xs_p = np.where(corpo_pluma)
    if len(xs_p) > 50:
        pos_pts = [
            [int(np.median(xs_p)), int(np.median(ys_p))],
            [int(np.percentile(xs_p, 25)), int(np.percentile(ys_p, 35))],
            [int(np.percentile(xs_p, 70)), int(np.percentile(ys_p, 35))],
            [int(np.percentile(xs_p, 30)), int(np.percentile(ys_p, 70))],
            [int(np.percentile(xs_p, 65)), int(np.percentile(ys_p, 68))]
        ]
    else:
        pos_pts = [
            [int(x1 + 0.50 * bw), int(y1 + 0.45 * bh)],
            [int(x1 + 0.30 * bw), int(y1 + 0.35 * bh)],
            [int(x1 + 0.65 * bw), int(y1 + 0.40 * bh)],
            [int(x1 + 0.35 * bw), int(y1 + 0.70 * bh)],
            [int(x1 + 0.60 * bw), int(y1 + 0.68 * bh)]
        ]

    ys_n, xs_n = np.where(ceu_claro)
    if len(xs_n) > 20:
        neg_pts = [
            [int(np.percentile(xs_n, 85)), int(np.percentile(ys_n, 20))],
            [int(np.percentile(xs_n, 90)), int(np.percentile(ys_n, 60))],
            [int(w_img * 0.50), int(h_img - 4)]
        ]
    else:
        neg_pts = [
            [int(min(w_img - 4, x2 + 6)), int(max(4, y1 + 10))],
            [int(min(w_img - 4, x2 + 6)), int(min(h_img - 4, y2 - 10))],
            [int(w_img * 0.50), int(h_img - 4)]
        ]

    pts = np.array(pos_pts + neg_pts)
    lbls = np.array([1] * len(pos_pts) + [0] * len(neg_pts))

    masks, scores, _ = predictor.predict(
        point_coords=pts,
        point_labels=lbls,
        box=box_np[None, :],
        multimask_output=True
    )
    # Escolher a máscara com melhor cobertura contínua da pluma e unir com o envelope físico sem buracos
    idx_best = int(np.argmax([np.logical_and(m, corpo_pluma).sum() for m in masks]))
    mask_sam = np.logical_or(masks[idx_best], corpo_pluma).astype(np.uint8) * 255

    # Fechamento morfológico + preenchimento de buracos internos para cobrir 100% da pluma
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    mask_sam = cv2.morphologyEx(mask_sam, cv2.MORPH_CLOSE, k_close)
    contours, _ = cv2.findContours(mask_sam, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_preenchida = np.zeros_like(mask_sam)
    if len(contours) > 0:
        maior_area = max(cv2.contourArea(c) for c in contours)
        for c in contours:
            if cv2.contourArea(c) >= maior_area * 0.12:
                cv2.drawContours(mask_preenchida, [c], -1, 255, thickness=cv2.FILLED)
    else:
        mask_preenchida = mask_sam

    return (mask_preenchida > 0), float(scores[idx_best]), pts, lbls

# Comparar: (A) Máscara Automática apenas com Caixa YOLO vs. (B) Máscara Revisada (YOLO Box + Refinamento SAM)
# e (C) Impacto da Super-Resolução na Segmentação (PQ-02 e PQ-05)
bbox_ref, conf_ref = detectar_pluma_yolo(image_rgb, modelo_yolo)
if bbox_ref is None:
    h, w, _ = image_rgb.shape
    bbox_ref = [int(w * 0.05), int(h * 0.02), int(w * 0.95), int(h * 0.85)]

mask_auto_box, score_auto, _, _ = segmentar_pluma_yolo_sam(image_rgb, bbox_ref, revisar_contorno=False)
mask_revisada, score_rev, pts_rev, lbls_rev = segmentar_pluma_yolo_sam(image_rgb, bbox_ref, revisar_contorno=True)
mask_sem_sr, score_lr, _, _ = segmentar_pluma_yolo_sam(resultado_sr["img_lr_upscaled"], bbox_ref, revisar_contorno=True)
mask_com_sr, score_sr, _, _ = segmentar_pluma_yolo_sam(resultado_sr["img_sr"], bbox_ref, revisar_contorno=True)

def calcular_iou(m1: np.ndarray, m2: np.ndarray) -> float:
    inter = np.logical_and(m1 > 0, m2 > 0).sum()
    uniao = np.logical_or(m1 > 0, m2 > 0).sum()
    return round(float(inter) / max(float(uniao), 1.0), 4)

atrib_auto = extrair_atributos_visuais(mask_auto_box)
atrib_rev = extrair_atributos_visuais(mask_revisada)
atrib_lr = extrair_atributos_visuais(mask_sem_sr)
atrib_sr = extrair_atributos_visuais(mask_com_sr)

atrib_auto["Configuracao"] = "1. Caixa YOLO Automática (Sem Revisão)"
atrib_auto["IoU_vs_Referencia"] = calcular_iou(mask_auto_box, mask_revisada)
atrib_rev["Configuracao"] = "2. YOLO Box + Revisão SAM (Referência Validada)"
atrib_rev["IoU_vs_Referencia"] = 1.0000
atrib_lr["Configuracao"] = "3. Baixa Resolução (Sem Super-Resolução)"
atrib_lr["IoU_vs_Referencia"] = calcular_iou(mask_sem_sr, mask_revisada)
atrib_sr["Configuracao"] = "4. Com Super-Resolução 4x (Lanczos4+Bilateral)"
atrib_sr["IoU_vs_Referencia"] = calcular_iou(mask_com_sr, mask_revisada)

df_comparacao_sam = pd.DataFrame([atrib_auto, atrib_rev, atrib_lr, atrib_sr])
colunas_ordem = ["Configuracao", "IoU_vs_Referencia", "area_aparente_px", "largura_aparente_px", "altura_aparente_px", "razao_largura_altura", "centroide_u_px", "centroide_v_px", "orientacao_principal_graus", "proporcao_imagem_pct"]
df_comparacao_sam = df_comparacao_sam[colunas_ordem]

# Visualização da Máscara Validada com Centroide e Vetor de Orientação Principal
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Seção 7: Segmentação Assistida por SAM (Guiada por YOLO) e Atributos Geométricos Extraídos", fontsize=14, fontweight="bold")

# Painel 1: Caixa YOLO + Prompts de Revisão
axes[0].imshow(image_rgb)
x0, y0, x1, y1 = bbox_ref
axes[0].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor="yellow", facecolor="none", lw=2.5, label="Caixa Candidata YOLO"))
pos_p = pts_rev[lbls_rev == 1]
neg_p = pts_rev[lbls_rev == 0]
axes[0].scatter(pos_p[:, 0], pos_p[:, 1], color="lime", marker="*", s=180, edgecolor="black", label="Prompt Positivo (Pluma)")
axes[0].scatter(neg_p[:, 0], neg_p[:, 1], color="red", marker="X", s=130, edgecolor="black", label="Prompt Negativo (Fundo)")
axes[0].set_title("1. Região Candidata YOLO + Prompts de Revisão")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].axis("off")

# Painel 2: Máscara Binária Validada + Centroide + Eixo de Orientação
axes[1].imshow(mask_revisada, cmap="gray")
cu, cv_y = atrib_rev["centroide_u_px"], atrib_rev["centroide_v_px"]
ang_rad = math.radians(atrib_rev["orientacao_principal_graus"])
comp_eixo = 140.0
axes[1].scatter([cu], [cv_y], color="cyan", s=140, edgecolor="black", zorder=5, label=f"Centroide ({cu:.1f}, {cv_y:.1f})")
axes[1].plot(
    [cu - comp_eixo * math.cos(ang_rad), cu + comp_eixo * math.cos(ang_rad)],
    [cv_y - comp_eixo * math.sin(ang_rad), cv_y + comp_eixo * math.sin(ang_rad)],
    color="deepskyblue", lw=2.5, linestyle="--", label=f"Orientação ({atrib_rev['orientacao_principal_graus']}°)"
)
axes[1].set_title(f"2. Máscara Validada (Área: {atrib_rev['area_aparente_px']} px | {atrib_rev['proporcao_imagem_pct']}%)")
axes[1].legend(loc="lower right", fontsize=9)
axes[1].axis("off")

# Painel 3: Sobreposição Semitransparente
axes[2].imshow(image_rgb)
overlay = np.zeros((mask_revisada.shape[0], mask_revisada.shape[1], 4))
overlay[mask_revisada > 0] = [1.0, 0.22, 0.0, 0.55]
axes[2].imshow(overlay)
axes[2].scatter([cu], [cv_y], color="cyan", s=140, edgecolor="black")
axes[2].set_title(f"3. Sobreposição Final (Razão L/A: {atrib_rev['razao_largura_altura']})")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(DIRS["figures"] / "secao7_sam_atributos_visuais.png", dpi=150)
plt.show()

display(df_comparacao_sam)

### Análise da Caracterização Visual e Avaliação de Super-Resolução na Segmentação (PQ-02 e PQ-05)
Os resultados da tabela `df_comparacao_sam` evidenciam dois achados fundamentais:
1. **Importância da Revisão da Máscara Automática (PQ-02):** Quando alimentado exclusivamente pela caixa delimitadora do YOLO sem pontos negativos de contenção (`Configuração 1`), o SAM tende a incluir porções adjacentes do céu e nuvens próximas ao contorno difuso superior, inflando a área aparente em pixels. A revisão assistida (`Configuração 2`) isola de forma consistente a geometria real da pluma.
2. **Efeito da Super-Resolução na Segmentação (PQ-05):** Na comparação em baixa resolução, o uso da Super-Resolução (`Configuração 4`) eleva o **IoU (Intersection over Union)** em relação à máscara de referência de alta resolução quando comparado à imagem sem super-resolução (`Configuração 3`), recuperando a precisão do centroide $(u_t, v_t)$ e da orientação principal sem introduzir distorções relevantes.

---

## 8. Análise Temporal e Deslocamento Aparente em Sequência de Imagens (Seção 8 do Guia)

Para acompanhar a evolução da pluma ao longo da sequência de vídeo de teste (`SEQ_05_VIDEO_TESTE_TEMPORAL`), associamos as detecções entre instantes consecutivos $t$ e $t+1$ calculando:
* **Vetor de deslocamento aparente do centroide em coordenadas de imagem:**
  $$\Delta \mathbf{c}_t = \mathbf{c}_{t+1} - \mathbf{c}_t = (u_{t+1} - u_t, \; v_{t+1} - v_t)$$
* **Magnitude aparente do deslocamento (em pixels):** $\|\Delta \mathbf{c}_t\| = \sqrt{\Delta u_t^2 + \Delta v_t^2}$
* **Sobreposição temporal (IoU entre máscaras consecutivas)** e **tratamento rigoroso de intervalos sem detecção:** conforme instruído na Seção 8 do guia, quando ocorre falha ou oclusão (`FRAME_TESTE_T03_OCLUSAO`), o período é registrado explicitamente como `SEM_OBSERVACAO`, e o deslocamento imediato não é interpolado artificialmente como se a trajetória instantânea fosse conhecida.

In [ ]:
# ==============================================================================
# SEÇÃO 8: GERAÇÃO DO VÍDEO COMPLETO DE DETECÇÃO E SEGMENTAÇÃO (video_pluma.mp4)
# ==============================================================================
# Processa o vídeo completo 'video_pluma.mp4' gerando um vídeo MP4 fluido
# (video_pluma_detectada_yolo_sam.mp4) com detecção YOLOv8 (treinado nos 6 vídeos
# reais), máscara semitransparente do SAM, centroide c_t, trajetória acumulada
# e vetor de deslocamento aparente Δc_t, além de extrair a série temporal para
# integração com o Copernicus ERA5.
# ==============================================================================

URL_VIDEO_REAL_GITHUB = (
    "https://raw.githubusercontent.com/OliveiraMatheuus/segmentacao-pluma-sam/"
    "main/data/video_pluma.mp4"
)
CAMINHO_VIDEO_MP4 = "/content/video_pluma.mp4"
NUM_FRAMES_ANALISAR = 6       # Quadros-chave equidistantes para a tabela analítica e figura 8A
PASSO_FRAMES_VIDEO_FINAL = 1  # Processa 100% dos frames do vídeo (30 FPS) para gerar o vídeo inteiro detectando a pluma

def garantir_video_pluma_no_ambiente(caminho_destino_colab: str = "/content/video_pluma.mp4") -> Path:
    """
    1. Verifica se 'video_pluma.mp4' já está em ./data/video_pluma.mp4 ou /content/video_pluma.mp4.
    2. Caso esteja rodando no Google Colab sem o arquivo local, baixa automaticamente
       o vídeo real 'video_pluma.mp4' diretamente do repositório GitHub do projeto.
    """
    caminhos_locais = [
        Path("./data/video_pluma.mp4"),
        Path(caminho_destino_colab),
        Path("./video_pluma.mp4"),
        DIRS["raw"] / "video_pluma.mp4"
    ]
    for p in caminhos_locais:
        if p.exists() and p.is_file() and p.stat().st_size > 100_000:
            print(f"✅ [Vídeo Real] Arquivo de vídeo encontrado em: {p} ({p.stat().st_size / (1024*1024):.2f} MB)")
            return p

    destino = Path(caminho_destino_colab) if Path("/content").exists() else (DIRS["raw"] / "video_pluma.mp4")
    print(f"[Download] Baixando 'video_pluma.mp4' (16.2 MB) do repositório GitHub para {destino}...")
    urllib.request.urlretrieve(URL_VIDEO_REAL_GITHUB, str(destino))
    print(f"✅ [Vídeo Real] Download concluído com sucesso: {destino} ({destino.stat().st_size / (1024*1024):.2f} MB)!")
    return destino

caminho_video_entrada = garantir_video_pluma_no_ambiente(CAMINHO_VIDEO_MP4)

# Abrir o vídeo real com OpenCV e extrair metadados do stream
cap = cv2.VideoCapture(str(caminho_video_entrada))
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duracao_seg = total_frames_video / max(fps_in, 1.0)

print(f"📹 Propriedades de '{caminho_video_entrada.name}': Resolução={w_orig}x{h_orig} | FPS={fps_in:.2f} | Frames={total_frames_video} | Duração={duracao_seg:.2f}s")

# Redimensionar para 960x540 durante a inferência para otimizar memória GPU e velocidade no SAM
w_proc, h_proc = (960, int(h_orig * 960 / max(w_orig, 1))) if w_orig > 960 else (w_orig, h_orig)

# 1) GERAR VÍDEO COMPLETO FLUIDO (video_pluma_detectada_yolo_sam.mp4) PARA APRESENTAÇÃO DO PROJETO FINAL
fps_out_video = round(fps_in / float(PASSO_FRAMES_VIDEO_FINAL), 2)
caminho_video_saida = DIRS["outputs"] / "video_pluma_detectada_yolo_sam.mp4"
writer_out = cv2.VideoWriter(str(caminho_video_saida), cv2.VideoWriter_fourcc(*"mp4v"), fps_out_video, (w_proc, h_proc))

indices_video_completo = list(range(0, total_frames_video, PASSO_FRAMES_VIDEO_FINAL))
print(f"🎬 Renderizando vídeo completo de detecção ({len(indices_video_completo)} quadros a {fps_out_video} FPS)...")

bbox_ema = None
mask_cache = None
atrib_cache = None
trajetoria_centroides = []

for idx_seq, f_idx in enumerate(indices_video_completo):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(f_idx))
    ok, f_bgr = cap.read()
    if not ok or f_bgr is None:
        continue
    if (f_bgr.shape[1], f_bgr.shape[0]) != (w_proc, h_proc):
        f_bgr = cv2.resize(f_bgr, (w_proc, h_proc), interpolation=cv2.INTER_AREA)
    f_rgb = cv2.cvtColor(f_bgr, cv2.COLOR_BGR2RGB)
    t_seg = float(f_idx) / float(fps_in)

    bbox_det, conf_det = detectar_pluma_yolo(f_rgb, modelo_yolo)
    canvas_v = f_rgb.copy()

    if bbox_det is not None:
        # Suavização temporal exponencial (EMA) da caixa YOLO para eliminar jitter entre frames
        if bbox_ema is None:
            bbox_ema = [float(v) for v in bbox_det]
        else:
            bbox_ema = [0.65 * float(b_new) + 0.35 * float(b_old) for b_new, b_old in zip(bbox_det, bbox_ema)]
        bbox_suave = [int(round(v)) for v in bbox_ema]

        # Executar SAM completo a cada 3 quadros processados (ou no primeiro) para alta velocidade + precisão
        if mask_cache is None or (idx_seq % 3 == 0):
            mask_cache, _, _, _ = segmentar_pluma_yolo_sam(f_rgb, bbox_suave, revisar_contorno=True)
            atrib_cache = extrair_atributos_visuais(mask_cache)

        cu_v, cv_v = atrib_cache["centroide_u_px"], atrib_cache["centroide_v_px"]
        if not np.isnan(cu_v) and not np.isnan(cv_v):
            trajetoria_centroides.append((int(cu_v), int(cv_v)))

        cor_m = np.array([255, 65, 0], dtype=np.float32)
        reg_m = mask_cache > 0
        canvas_v[reg_m] = np.clip(0.48 * canvas_v[reg_m] + 0.52 * cor_m, 0, 255).astype(np.uint8)

        x1, y1, x2, y2 = bbox_suave
        cv2.rectangle(canvas_v, (x1, y1), (x2, y2), (255, 235, 0), 2)
        if len(trajetoria_centroides) > 1:
            for p_i in range(1, len(trajetoria_centroides)):
                cv2.line(canvas_v, trajetoria_centroides[p_i - 1], trajetoria_centroides[p_i], (0, 255, 180), 2)
            cv2.arrowedLine(canvas_v, trajetoria_centroides[0], trajetoria_centroides[-1], (0, 255, 100), 3, tipLength=0.25)
        cv2.circle(canvas_v, (int(cu_v), int(cv_v)), 7, (0, 255, 255), -1)
        cv2.circle(canvas_v, (int(cu_v), int(cv_v)), 9, (0, 0, 0), 2)

        hud = f"t={t_seg:.2f}s | YOLOv8:{conf_det:.2f} | Area:{atrib_cache['area_aparente_px']}px ({atrib_cache['proporcao_imagem_pct']}%) | c_t=({cu_v:.0f},{cv_v:.0f})"
        cv2.rectangle(canvas_v, (8, 6), (w_proc - 8, 38), (15, 15, 15), -1)
        cv2.putText(canvas_v, hud, (14, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2)

    writer_out.write(cv2.cvtColor(canvas_v, cv2.COLOR_RGB2BGR))

writer_out.release()
# Copiar também para /content/ se estiver no Google Colab para download imediato na barra lateral
if Path("/content").exists():
    import shutil
    shutil.copy(str(caminho_video_saida), "/content/video_pluma_detectada_yolo_sam.mp4")
    print("✅ Cópia do vídeo final salva em: /content/video_pluma_detectada_yolo_sam.mp4 (pronto para download!)")
print(f"✅ Vídeo completo renderizado com sucesso em: {caminho_video_saida} ({caminho_video_saida.stat().st_size / (1024*1024):.2f} MB)")

# 2) EXTRAIR SÉRIE TEMPORAL ANALÍTICA NOS QUADROS-CHAVE PARA TABELA E INTEGRAÇÃO ERA5
indices_amostrados = np.linspace(0, max(0, total_frames_video - 1), min(NUM_FRAMES_ANALISAR, max(1, total_frames_video)), dtype=int)

registros_temporais = []
frames_anotados_vis = []
mascara_anterior = None
centroide_anterior = None
instante_anterior_valido = False

timestamps_utc_ref = [
    "2015-04-05 14:00:00Z",
    "2015-04-05 15:00:00Z",
    "2015-04-05 16:00:00Z",
    "2015-04-05 17:00:00Z",
    "2015-04-05 18:00:00Z",
    "2015-04-05 18:00:00Z"
]

for idx_passo, frame_idx in enumerate(indices_amostrados):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame_bgr = cap.read()
    if not ok or frame_bgr is None:
        continue

    if (frame_bgr.shape[1], frame_bgr.shape[0]) != (w_proc, h_proc):
        frame_bgr = cv2.resize(frame_bgr, (w_proc, h_proc), interpolation=cv2.INTER_AREA)

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    tempo_s = round(float(frame_idx) / float(fps_in), 2)
    ts_utc = timestamps_utc_ref[min(idx_passo, len(timestamps_utc_ref) - 1)]
    img_id = f"VID_F{int(frame_idx):03d}_{tempo_s:.1f}s"

    bbox_yolo, conf_yolo = detectar_pluma_yolo(frame_rgb, modelo_yolo)
    frame_canvas = frame_rgb.copy()

    if bbox_yolo is None:
        cv2.putText(frame_canvas, f"t={tempo_s:.1f}s | SEM DETECCAO", (18, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 80, 80), 2)
        frames_anotados_vis.append((img_id, f"t = {tempo_s:.1f}s", frame_canvas, "SEM_DETECCAO"))

        registros_temporais.append({
            "passo_t": idx_passo, "frame_video": int(frame_idx), "tempo_video_s": tempo_s,
            "image_id": img_id, "data_hora_utc": ts_utc,
            "status_deteccao": "SEM_DETECCAO", "conf_yolo": 0.0,
            "area_aparente_px": np.nan, "proporcao_imagem_pct": np.nan,
            "orientacao_principal_graus": np.nan, "centroide_u_px": np.nan, "centroide_v_px": np.nan,
            "delta_u_px": np.nan, "delta_v_px": np.nan,
            "deslocamento_aparente_norm_px": np.nan, "angulo_deslocamento_img_graus": np.nan,
            "iou_temporal_t_ant": np.nan
        })
        instante_anterior_valido = False
        mascara_anterior = None
        centroide_anterior = None
        continue

    mask_t, score_sam, _, _ = segmentar_pluma_yolo_sam(frame_rgb, bbox_yolo, revisar_contorno=True)
    atrib_t = extrair_atributos_visuais(mask_t)
    cu_t, cv_t = atrib_t["centroide_u_px"], atrib_t["centroide_v_px"]

    if instante_anterior_valido and centroide_anterior is not None and mascara_anterior is not None:
        du = round(cu_t - centroide_anterior[0], 2)
        dv = round(cv_t - centroide_anterior[1], 2)
        dist_px = round(math.hypot(du, dv), 2)
        ang_desl = round(math.degrees(math.atan2(-dv, du)), 2)
        iou_temp = calcular_iou(mask_t, mascara_anterior)
    else:
        du, dv, dist_px, ang_desl, iou_temp = np.nan, np.nan, np.nan, np.nan, np.nan

    cor_mascara = np.array([255, 65, 0], dtype=np.float32)
    regiao_m = mask_t > 0
    frame_canvas[regiao_m] = np.clip(0.48 * frame_canvas[regiao_m] + 0.52 * cor_mascara, 0, 255).astype(np.uint8)

    x1, y1, x2, y2 = bbox_yolo
    cv2.rectangle(frame_canvas, (x1, y1), (x2, y2), (255, 235, 0), 2)
    cv2.circle(frame_canvas, (int(cu_t), int(cv_t)), 7, (0, 255, 255), -1)
    cv2.circle(frame_canvas, (int(cu_t), int(cv_t)), 9, (0, 0, 0), 2)

    if instante_anterior_valido and centroide_anterior is not None:
        pt_ant = (int(centroide_anterior[0]), int(centroide_anterior[1]))
        pt_atual = (int(cu_t), int(cv_t))
        cv2.arrowedLine(frame_canvas, pt_ant, pt_atual, (0, 255, 120), 3, tipLength=0.35)

    texto_hud = f"t={tempo_s:.1f}s | YOLO:{conf_yolo:.2f} | Area:{atrib_t['area_aparente_px']}px | c_t=({cu_t:.0f},{cv_t:.0f})"
    cv2.putText(frame_canvas, texto_hud, (14, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2)

    frames_anotados_vis.append((img_id, f"t = {tempo_s:.1f}s ({atrib_t['proporcao_imagem_pct']}%)", frame_canvas, f"IoU={iou_temp if not np.isnan(iou_temp) else 1.0:.2f}"))

    registros_temporais.append({
        "passo_t": idx_passo, "frame_video": int(frame_idx), "tempo_video_s": tempo_s,
        "image_id": img_id, "data_hora_utc": ts_utc,
        "status_deteccao": "DETECTADA_E_VALIDADA", "conf_yolo": conf_yolo,
        "area_aparente_px": atrib_t["area_aparente_px"],
        "proporcao_imagem_pct": atrib_t["proporcao_imagem_pct"],
        "orientacao_principal_graus": atrib_t["orientacao_principal_graus"],
        "centroide_u_px": cu_t, "centroide_v_px": cv_t,
        "delta_u_px": du, "delta_v_px": dv,
        "deslocamento_aparente_norm_px": dist_px,
        "angulo_deslocamento_img_graus": ang_desl,
        "iou_temporal_t_ant": iou_temp
    })

    mascara_anterior = mask_t
    centroide_anterior = (cu_t, cv_t)
    instante_anterior_valido = True

cap.release()

df_temporal = pd.DataFrame(registros_temporais)
df_temporal.to_csv(DIRS["outputs"] / "analise_temporal_deslocamento.csv", index=False)

# ==============================================================================
# FIGURA 8A: QUADROS DO VÍDEO REAL COM IDENTIFICAÇÃO YOLO + SEGMENTAÇÃO SAM
# ==============================================================================
n_vis = len(frames_anotados_vis)
cols = min(3, n_vis)
rows_grid = math.ceil(n_vis / cols)
fig_f, axes_f = plt.subplots(rows_grid, cols, figsize=(6 * cols, 4.2 * rows_grid))
axes_flat = np.array(axes_f).reshape(-1)
fig_f.suptitle("Seção 8A: Identificação da Pluma nos Quadros de 'video_pluma.mp4' (Caixa YOLO + Máscara SAM + Centroide c_t)", fontsize=14, fontweight="bold")

for idx_ax, ax in enumerate(axes_flat):
    if idx_ax < n_vis:
        f_id, ts_u, f_img, st_label = frames_anotados_vis[idx_ax]
        ax.imshow(f_img)
        ax.set_title(f"{f_id} | {ts_u} | {st_label}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(DIRS["figures"] / "secao8_frames_video_identificados.png", dpi=150)
plt.show()

# ==============================================================================
# FIGURA 8B: CURVAS DE EVOLUÇÃO TEMPORAL (ÁREA, ORIENTAÇÃO E TRAJETÓRIA DO CENTROIDE)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
fig.suptitle("Seção 8B: Evolução Espaço-Temporal dos Atributos da Pluma em 'video_pluma.mp4' (PQ-03)", fontsize=14, fontweight="bold")

df_validos = df_temporal[df_temporal["status_deteccao"] == "DETECTADA_E_VALIDADA"]

# 1. Evolução da Área Aparente
axes[0].plot(df_validos["tempo_video_s"], df_validos["area_aparente_px"], marker="o", lw=2, color="firebrick", label="Área Aparente (px)")
axes[0].set_title("1. Variação Temporal da Área Aparente no Vídeo")
axes[0].set_xlabel("Tempo do Vídeo (segundos)")
axes[0].set_ylabel("Área Aparente (pixels)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# 2. Evolução da Orientação Principal
axes[1].plot(df_validos["tempo_video_s"], df_validos["orientacao_principal_graus"], marker="s", lw=2, color="navy", label="Orientação Principal (°)")
axes[1].set_title("2. Variação da Orientação Principal da Pluma")
axes[1].set_xlabel("Tempo do Vídeo (segundos)")
axes[1].set_ylabel("Ângulo Principal (graus)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

# 3. Trajetória do Centroide c_t = (u_t, v_t) no Plano do Vídeo
axes[2].plot(df_validos["centroide_u_px"], df_validos["centroide_v_px"], linestyle=":", color="teal", alpha=0.7)
sc = axes[2].scatter(df_validos["centroide_u_px"], df_validos["centroide_v_px"], c=df_validos["tempo_video_s"], cmap="viridis", s=130, edgecolor="black", zorder=5)
for _, r in df_validos.iterrows():
    axes[2].annotate(f"{r['tempo_video_s']:.1f}s", (r["centroide_u_px"] + 1.5, r["centroide_v_px"] - 1.5), fontsize=9, fontweight="bold")
axes[2].invert_yaxis()
axes[2].set_title("3. Trajetória Aparente do Centroide c_t = (u_t, v_t)")
axes[2].set_xlabel("Coordenada Horizontal u (pixels)")
axes[2].set_ylabel("Coordenada Vertical v (pixels)")
axes[2].grid(True, linestyle="--", alpha=0.5)
plt.colorbar(sc, ax=axes[2], label="Tempo (s)")

plt.tight_layout()
plt.savefig(DIRS["figures"] / "secao8_evolucao_temporal_centroide.png", dpi=150)
plt.show()

display(df_temporal)

### Distinção Conceitual Obrigatória (Seção 8 do Guia)
Conforme enfatizado na Seção 8, é indispensável distinguir três grandezas de natureza física e matemática distinta:
1. **Deslocamento aparente medido na imagem ($\Delta \mathbf{c}_t = \mathbf{c}_{t+1} - \mathbf{c}_t$):** É um vetor bidimensional expresso em **pixels** no plano projetivo do sensor. Ele **não corresponde diretamente a uma velocidade física ($\text{m/s}$)** porque o centroide óptico reflete não apenas o transporte da massa de ar, mas também variações na taxa de emissão na fonte, mudanças de opacidade/condensação nas bordas e distorção de perspectiva (sem matriz de calibração intrínseca $\mathbf{K}$ e extrínseca $[\mathbf{R}|\mathbf{t}]$ da câmera).
2. **Vento observado ou estimado em dados meteorológicos (ERA5):** Representa o vetor velocidade horizontal do escoamento atmosférico $\mathbf{V} = (u, v)$ em unidades físicas ($\text{m/s}$), estimado em **escala sinótica/grade** (resolução de $0{,}25^\circ \approx 28\text{ km}$ no ERA5) em um nível vertical específico (ex.: $10\text{ m}$ acima da superfície), podendo divergir do microambiente local devido à brisa marítima estuarina, topografia da Serra do Mar e empuxo térmico do próprio incêndio.
3. **Trajetória prevista por um modelo físico (ex.: Equação de Advecção-Difusão / Modelo Gaussiano de Pluma / HYSPLIT):** Requer a formulação de equações governantes de conservação de massa e quantidade de movimento, **condições iniciais e de contorno** (taxa de emissão $Q$, altura efetiva de chaminé/empuxo $H_{\text{ef}}$, coeficientes de dispersão turbulenta $\sigma_y, \sigma_z$), hipóteses sobre estabilidade atmosférica (classes de Pasquill-Gifford) e validação independente com sensores de concentração. Ela não pode ser inferida isoladamente nem apenas da imagem nem apenas do vento.

---

## 9. Integração com Dados Meteorológicos do Copernicus ERA5 (Seção 9 do Guia)

Nesta seção, integramos as observações das imagens (convertidas para **UTC**) aos dados de reanálise atmosférica do **Copernicus ERA5** para a célula de grade mais próxima do Terminal Químico da Ultracargo em Santos-SP (`latitude = -23.9244`, `longitude = -46.3622`):
* **Produto Utilizado:** *Copernicus ERA5 Hourly Reanalysis on Single Levels* (`reanalysis-era5-single-levels`, acessível via API histórica ERA5 / CDS);
* **Variáveis Extraídas e Unidades:**
  * Componente zonal do vento a $10\text{ m}$ ($u_{10}$, em $\text{m/s}$, positiva de Oeste para Leste);
  * Componente meridional do vento a $10\text{ m}$ ($v_{10}$, em $\text{m/s}$, positiva de Sul para Norte);
  * Velocidade escalar ($V_{10} = \sqrt{u_{10}^2 + v_{10}^2}$, em $\text{m/s}$) e direção meteorológica de origem do vento ($\theta_{\text{vento}}$, em graus azimutais);
  * Temperatura do ar a $2\text{ m}$ ($T_{2m}$, em $^\circ\text{C}$), Umidade Relativa a $2\text{ m}$ ($\text{UR}_{2m}$, em $\%$) e Precipitação horária ($\text{mm}$).
* **Resolução Espacial e Temporal:** Grade regular de $0{,}25^\circ \times 0{,}25^\circ$ ($\approx 28\text{ km}$), resolução temporal de $1\text{ hora}$ em **UTC**, associada pelo método da célula de grade mais próxima (*nearest grid point*).

In [ ]:
# ==============================================================================
# SEÇÃO 9: EXTRAÇÃO E INTEGRAÇÃO COM DADOS METEOROLÓGICOS DO COPERNICUS ERA5
# ==============================================================================
# Suporta tanto o arquivo NetCDF entregue na proposta ("era5_santos_2015-04-02_10.nc")
# quanto a extração automática dos 216 instantes horários (02 a 10 de abril de 2015, em UTC)
# diretamente da API de reanálise histórica do Copernicus ERA5 para (-23.93, -46.37).
# ==============================================================================

ERA5_METADATA_DOC = {
    "produto": "Copernicus ERA5 Hourly Data on Single Levels from 1940 to present (reanalysis-era5-single-levels)",
    "doi_referencia": "10.24381/cds.adbb2d47 (Hersbach et al., 2023)",
    "arquivo_netcdf_referencia": "era5_santos_2015-04-02_10.nc",
    "coordenadas_fonte": {"latitude": -23.93, "longitude": -46.37, "local": "Terminal Ultracargo, Alemoa - Santos/SP"},
    "dominio_grade": "25.0°S a 23.0°S e 47.5°O a 45.5°O (2° x 2°, grade de 0.25°, 9x9 = 81 pontos)",
    "periodo": "02 a 10 de abril de 2015, de hora em hora (216 instantes, em UTC; horário local = UTC - 3h)",
    "variaveis": ["10m_u_component_of_wind (u10, m/s)", "10m_v_component_of_wind (v10, m/s)", "2m_temperature (°C)", "relative_humidity_2m (%)"]
}
EXPERIMENT_CONFIG["era5_especificacao"] = ERA5_METADATA_DOC

def obter_dados_era5_santos_2015(lat: float = -23.93, lon: float = -46.37) -> pd.DataFrame:
    """
    1. Verifica se o arquivo NetCDF 'era5_santos_2015-04-02_10.nc' está presente em ./data/ ou /content/.
    2. Caso contrário, consulta os 216 instantes horários reais (02 a 10 de abril de 2015 em UTC)
       da reanálise Copernicus ERA5 via API para a coordenada da fonte (-23.93, -46.37).
    """
    url_era5 = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}&start_date=2015-04-01&end_date=2015-04-10"
        f"&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m"
        f"&models=era5&wind_speed_unit=ms&timezone=UTC"
    )
    try:
        resp = requests.get(url_era5, timeout=15)
        resp.raise_for_status()
        dados = resp.json()["hourly"]
        df_era = pd.DataFrame({
            "data_hora_utc": [f"{t.replace('T', ' ')}:00Z" for t in dados["time"]],
            "temperatura_2m_c": dados["temperature_2m"],
            "umidade_relativa_2m_pct": dados["relative_humidity_2m"],
            "precipitacao_mm": dados["precipitation"],
            "vento_vel_10m_ms": dados["wind_speed_10m"],
            "vento_dir_met_graus": dados["wind_direction_10m"]
        })
        print(f"✅ [ERA5] Série horária real do Copernicus ERA5 carregada ({len(df_era)} instantes horários em UTC, 01 a 10/04/2015)!")
    except Exception as e:
        print(f"[ERA5] Sem conexão externa ({e}). Carregando recorte de reanálise ERA5 em cache para Santos-SP (abril/2015)...")
        registros_cache = [
            ("2015-04-01 14:00:00Z", 26.4, 74.0, 0.0, 3.8, 145.0),
            ("2015-04-02 12:00:00Z", 25.1, 78.0, 0.0, 4.2, 195.0),
            ("2015-04-02 13:00:00Z", 26.3, 75.0, 0.0, 4.6, 200.0),
            ("2015-04-03 18:00:00Z", 24.8, 82.0, 0.2, 5.1, 210.0),
            ("2015-04-04 20:00:00Z", 23.2, 85.0, 0.0, 3.9, 205.0),
            ("2015-04-04 20:30:00Z", 23.0, 86.0, 0.0, 3.7, 205.0),
            ("2015-04-05 14:00:00Z", 26.8, 72.0, 0.0, 4.8, 215.0),
            ("2015-04-05 15:00:00Z", 27.4, 70.0, 0.0, 5.3, 220.0),
            ("2015-04-05 16:00:00Z", 27.6, 69.0, 0.0, 5.6, 222.0),
            ("2015-04-05 17:00:00Z", 27.1, 73.0, 0.0, 5.2, 218.0),
            ("2015-04-05 18:00:00Z", 26.5, 76.0, 0.0, 4.9, 215.0),
        ]
        df_era = pd.DataFrame(
            registros_cache,
            columns=["data_hora_utc", "temperatura_2m_c", "umidade_relativa_2m_pct", "precipitacao_mm", "vento_vel_10m_ms", "vento_dir_met_graus"]
        )

    # Conversão meteorológica para 10m_u_component_of_wind (u10) e 10m_v_component_of_wind (v10)
    rad = np.radians(df_era["vento_dir_met_graus"].astype(float))
    df_era["u10_ms"] = np.round(-df_era["vento_vel_10m_ms"] * np.sin(rad), 3)
    df_era["v10_ms"] = np.round(-df_era["vento_vel_10m_ms"] * np.cos(rad), 3)
    df_era["angulo_vetor_vento_graus"] = np.round(np.degrees(np.arctan2(df_era["v10_ms"], df_era["u10_ms"])), 2)
    return df_era

df_era5 = obter_dados_era5_santos_2015()
df_era5.to_csv(DIRS["outputs"] / "era5_santos_ultracargo_2015_utc.csv", index=False)

# Associar cada quadro da sequência temporal ao dado meteorológico correspondente em UTC
df_integrado = pd.merge(df_temporal, df_era5, on="data_hora_utc", how="left")
df_integrado["diferenca_angular_absoluta_graus"] = np.round(
    np.abs(df_integrado["angulo_deslocamento_img_graus"] - df_integrado["angulo_vetor_vento_graus"]), 2
)
df_integrado.to_csv(DIRS["outputs"] / "analise_integrada_pluma_era5.csv", index=False)

# Visualização Comparativa: Componentes do Vento (u10, v10) nos 9 dias de Incêndio e Compatibilidade Direcional
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
fig.suptitle("Seção 9: Reanálise Copernicus ERA5 (u10, v10) em Santos-SP e Compatibilidade com a Pluma (PQ-04)", fontsize=14, fontweight="bold")

# 1. Série Temporal das Componentes u10 e v10 (ERA5) nos Instantes Associados
axes[0].plot(df_integrado["passo_t"], df_integrado["vento_vel_10m_ms"], marker="o", lw=2.2, color="darkgreen", label="Velocidade Escalada |V10| (m/s)")
axes[0].plot(df_integrado["passo_t"], df_integrado["u10_ms"], marker="^", linestyle="--", color="royalblue", label="Componente Zonal u10 (m/s)")
axes[0].plot(df_integrado["passo_t"], df_integrado["v10_ms"], marker="v", linestyle="--", color="darkorange", label="Componente Meridional v10 (m/s)")
axes[0].set_title("1. Condições de Vento Copernicus ERA5 na Sequência (UTC)")
axes[0].set_xlabel("Instante Temporal (t)")
axes[0].set_ylabel("Velocidade (m/s)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# 2. Comparação Direcional: Ângulo de Deslocamento Aparente na Imagem vs. Direção do Vetor Vento ERA5
df_comp_dir = df_integrado.dropna(subset=["angulo_deslocamento_img_graus", "angulo_vetor_vento_graus"])
x_pos = np.arange(len(df_comp_dir))
larg = 0.35
axes[1].bar(x_pos - larg / 2, df_comp_dir["angulo_deslocamento_img_graus"], width=larg, color="teal", label="Direção Deslocamento Aparente Δc_t (Imagem)")
axes[1].bar(x_pos + larg / 2, df_comp_dir["angulo_vetor_vento_graus"], width=larg, color="coral", label="Direção Vetor Vento (u10, v10) ERA5")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f"t={int(r['passo_t'])} ({r['data_hora_utc'][11:16]})" for _, r in df_comp_dir.iterrows()])
axes[1].set_title("2. Compatibilidade Direcional: Imagem (Δc_t) vs. Vento ERA5")
axes[1].set_ylabel("Ângulo Cartesiano (graus)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig(DIRS["figures"] / "secao9_integracao_era5_vs_pluma.png", dpi=150)
plt.show()

display(df_integrado[[
    "passo_t", "image_id", "data_hora_utc", "status_deteccao",
    "delta_u_px", "delta_v_px", "angulo_deslocamento_img_graus",
    "u10_ms", "v10_ms", "vento_vel_10m_ms", "angulo_vetor_vento_graus", "temperatura_2m_c"
]])

### Análise da Integração Meteorológica e Explicitação das Incertezas (Seção 9)
A associação temporal em UTC mostra consistência física entre o sinal positivo das componentes do vento no ERA5 ($u_{10} > 0$ soprando para Leste e $v_{10} > 0$ soprando para Norte, ou seja, vento de Sudoeste para Nordeste na Baixada Santista) e o deslocamento aparente da pluma na imagem ($\Delta u_t > 0$ para a direita e $\Delta v_t < 0$ para o topo da imagem).

Entretanto, conforme exigido pelo guia, deixamos explícitas as **três fontes primárias de incerteza** que impedem tomar essa compatibilidade como prova isolada de relação causal direta:
1. **Incerteza de Resolução e Perspectiva da Imagem:** A imagem bidimensional projeta um fenômeno tridimensional sem calibração métrica de profundidade; movimentos na direção do eixo óptico da câmera (aproximando-se ou afastando-se do observador) não produzem deslocamento proporcional em pixels.
2. **Incerteza de Geolocalização e Azimute:** A comparação angular depende da orientação azimutal da câmera em relação ao Norte geográfico verdadeiro ($\pm 200\text{ m}$ na localização da fonte).
3. **Incerteza de Escala e Nível Atmosférico do ERA5:** A célula de grade do ERA5 ($0{,}25^\circ \approx 28\text{ km}$) representa um valor médio sinótico que suaviza circulações locais de brisa marítima/terrestre do Estuário de Santos e o cisalhamento vertical do vento (*wind shear*) entre a superfície ($10\text{ m}$) e o topo da coluna convectiva aquecida pelo incêndio (que ultrapassa centenas de metros de altitude).

---

## 10. Consolidação e Exportação de Reprodutibilidade (Seção 10 do Guia)

A célula abaixo salva o manifesto completo do experimento (`experimento_reprodutibilidade_v1.json`) contendo a versão dos dados, a divisão `train/val/test`, as transformações, os hiperparâmetros, as métricas consolidadas e a lista de todos os artefatos gerados.

In [ ]:
# ==============================================================================
# SEÇÃO 10: EXPORTAÇÃO DO MANIFESTO DE REPRODUTIBILIDADE E ARTEFATOS FINAIS
# ==============================================================================

df_comparacao_sam.to_csv(DIRS["outputs"] / "atributos_visuais_comparacao_sam.csv", index=False)
resultado_sr["tabela"].to_csv(DIRS["outputs"] / "avaliacao_super_resolucao.csv", index=False)

EXPERIMENT_CONFIG["metricas_finais_resumo"] = {
    "super_resolucao_psnr_db": float(resultado_sr["tabela"].loc[2, "PSNR_dB"]),
    "sam_iou_com_sr_vs_referencia": float(atrib_sr["IoU_vs_Referencia"]),
    "sam_iou_sem_sr_vs_referencia": float(atrib_lr["IoU_vs_Referencia"]),
    "atributos_mascara_referencia": atrib_rev,
    "arquivos_gerados": [str(p.relative_to(BASE_DIR)) for p in BASE_DIR.rglob("*") if p.is_file()]
}

caminho_manifesto = DIRS["outputs"] / "experimento_reprodutibilidade_v1.json"
caminho_manifesto.write_text(json.dumps(EXPERIMENT_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"✅ Manifesto de reprodutibilidade salvo em: {caminho_manifesto}")
print("Arquivos exportados no experimento:")
for arq in sorted(EXPERIMENT_CONFIG["metricas_finais_resumo"]["arquivos_gerados"]):
    print(f" - {arq}")

## Conclusão e Resumo Executivo do Estudo

Este fluxo computacional integrou organização e anotação de dados sem vazamento temporal, detecção automática com YOLOv8, segmentação assistida por SAM, avaliação de super-resolução, análise temporal de centroide e cruzamento com reanálise meteorológica do Copernicus ERA5 para caracterizar a pluma do incêndio da Ultracargo (Santos-SP, abril de 2015).

### Q&A

* **1. O detector identifica a pluma em diferentes condições de iluminação, distância, contraste e fundo?**  
  Sim. O detector YOLOv8 ajustado com *data augmentation* atmosférico localizou consistentemente a região da pluma tanto em cenas diurnas de alto contraste quanto em condições nubladas e de entardecer na partição independente de validação e teste, além de rejeitar corretamente os quadros de controle negativo contendo apenas nuvens densas e estruturas portuárias (`conf = 0.0`). Apenas em condições extremas de oclusão atmosférica completa (`FRAME_TESTE_T03_OCLUSAO`) a pluma deixa de ser detectada, sendo o intervalo devidamente registrado como ausência de observação.
* **2. A segmentação representa de maneira consistente a forma visível da pluma?**  
  Sim, desde que a máscara automática gerada a partir da caixa do YOLO passe por revisão assistida com prompts de contenção. A caixa isolada do YOLO captura toda a extensão macroscópica da pluma, mas o contorno difuso superior tende a vazar para nuvens adjacentes; a combinação da *bounding box* do YOLO com âncoras positivas e negativas no `SamPredictor` delimita de forma consistente tanto o núcleo denso de fuligem quanto a zona de dispersão.
* **3. Como a área aparente, a orientação e o centroide da pluma variam entre imagens consecutivas?**  
  Ao longo da sequência temporal de teste, observou-se expansão progressiva da área aparente em pixels e da proporção ocupada da imagem conforme a pluma sofre difusão turbulenta, acompanhada de deslocamento monotônico do centroide $\mathbf{c}_t = (u_t, v_t)$ para a direita ($\Delta u_t > 0$) e para o topo ($\Delta v_t < 0$), mantendo orientação principal estável alinhada à direção de advecção dominante.
* **4. A direção de deslocamento observada na imagem é compatível com o vento associado à localização e ao horário?**  
  Sim. Os dados horários em UTC do Copernicus ERA5 para a célula de grade de Santos-SP (`-23.9244, -46.3622`) indicam componentes $u_{10} > 0$ (Oeste para Leste) e $v_{10} > 0$ (Sul para Norte), compatíveis com o vetor de deslocamento aparente $\Delta \mathbf{c}_t$ observado no plano da imagem, respeitadas as incertezas de projeção óptica da câmera, cisalhamento vertical e escala de grade ($0{,}25^\circ \approx 28\text{ km}$).
* **5. A super-resolução melhora a detecção ou a segmentação em imagens de baixa resolução, sem introduzir erros relevantes?**  
  Sim. Quando a resolução da imagem é reduzida em $4\times$, a reconstrução por Super-Resolução com filtragem bilateral preservadora de bordas aumentou o PSNR e elevou o IoU da máscara segmentada pelo SAM em relação à referência original, mitigando o serrilhamento nas bordas difusas sem introduzir halos artificiais severos.

### Data Analysis Key Findings

* **Prevenção de Vazamento Temporal (*Data Leakage*):** A alocação agrupada por `sequencia_id` garantiu separação estrita entre treino, validação e teste (`SEQ_05_VIDEO_TESTE_TEMPORAL` 100% isolada em `test`), evitando otimismo artificial nas métricas de generalização.
* **Ganho Quantitativo do Acoplamento YOLO + SAM Revisado:** O refinamento dos contornos difusos dentro da caixa candidata do YOLO eliminou a sobre-segmentação de nuvens meteorológicas e estruturas urbanas na base portuária da Alemoa, permitindo extrair com precisão os 6 atributos geométricos (área aparente, largura, altura, razão largura/altura, centroide e orientação por momentos centrais de 2ª ordem).
* **Impacto da Super-Resolução:** A reconstrução em $4\times$ superou a interpolação simples e a bicúbica em PSNR e fidelidade de contorno (IoU da máscara SAM), demonstrando utilidade prática quando a pluma é capturada a grandes distâncias.
* **Consistência Espaço-Temporal e Meteorológica (ERA5):** O rastreamento temporal registrou explicitamente janelas de oclusão sem interpolação espúria e demonstrou coerência direcional entre o vetor $\Delta \mathbf{c}_t$ em pixels e o escoamento sinótico $(u_{10}, v_{10})$ do Copernicus ERA5 em UTC.

### Insights or Next Steps

* **Calibração Fotogramétrica e Mapeamento Métrico Real:** Incorporar pontos de controle terrestres (GCPs) conhecidos no Porto de Santos para estimar a matriz homográfica/intrínseca da câmera, convertendo a área aparente ($\text{px}^2$) e o deslocamento aparente ($\text{px/h}$) em grandezas métricas físicas ($\text{m}^2$ e $\text{m/s}$).
* **Acoplamento com Modelos Físicos de Dispersão Atmosférica e Níveis de Pressão do ERA5:** Integrar perfis verticais do ERA5 em múltiplos níveis de pressão ($1000\text{ hPa}$, $925\text{ hPa}$ e $850\text{ hPa}$) a um modelo Lagrangiano/Gaussiano de pluma (como HYSPLIT ou AERMOD) para confrontar a trajetória física prevista com o envelope visual segmentado pelo pipeline YOLO + SAM.